In [1]:
import numpy as np
import pandas as pd
import os
from joblib import Parallel, delayed
from copy import deepcopy
from dataclasses import dataclass
from dataclasses import dataclass, asdict
from tqdm import tqdm
from statistics import harmonic_mean
from typing import Dict, List, Any, Tuple
from metabci.brainda.datasets import Wang2016, BETA
from metabci.brainda.paradigms import SSVEP
from SUTLSSVEP import SUTLSSVEP
from metabci.brainda.algorithms.decomposition import SC_TRCA
from metabci.brainda.algorithms.decomposition import TtCCA
from metabci.brainda.algorithms.decomposition import TDCA
from metabci.brainda.algorithms.utils.model_selection import set_random_seeds
from metabci.brainda.algorithms.decomposition.base import generate_filterbank, generate_cca_references
from scipy.signal import butter
from scipy.stats import wilcoxon, t
from SQHAF import SQHAF
from iistlf import IISTLF
import torch
from EEGconformer import EEGConformer

In [2]:
#**************************************************
# Benchmark数据集读取处理
#**************************************************
Bench_dataset = Wang2016()
# 扩展到32个被试
Bench_subject_list = list(range(1, 36))

for s in Bench_subject_list:
    Bench_dataset.data_path(subject=s, path="/home/foam/metabci/mne_data")

events = Bench_dataset.events.keys()
freq_list = [str(Bench_dataset.get_freq(event)) for event in events]  # 获得所有刺激的频率

# ---------- Explicit experiment configuration ----------
t_start, t_end = 0.14, 1.14
sfreq = 250
window_length = t_end - t_start
n_samples = int(window_length * sfreq)

print(f"[Benchmark] Time window: {window_length:.2f}s ({n_samples} samples)")

# add 5-90Hz bandpass filter in raw hook
bandpass_low, bandpass_high = 5, 90
print(f"[Benchmark] Bandpass filter: {bandpass_low}-{bandpass_high} Hz")

def raw_hook(raw, caches):
    raw.filter(bandpass_low, bandpass_high, l_trans_bandwidth=2, h_trans_bandwidth=5, phase='zero-double')
    caches['raw_stage'] = caches.get('raw_stage', -1) + 1
    return raw, caches

# ------------------------------------------------------

Bench_paradigm = SSVEP(
    channels=['POZ', 'PZ', 'PO3', 'PO5', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],  # 选择电极通道
    intervals=[(t_start, t_end)],
    events=freq_list,
    srate=sfreq
)

Bench_paradigm.register_raw_hook(raw_hook)

all_subjects = list(range(1, 36))


--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yijun/S23.mat.7z
--------ssssss, /upload/yijun/S24.mat.7z
--------ssssss, /upload/y

In [9]:
# ============================================================
# 1) Utilities
# ============================================================
def make_harmonic_filterbank(
    srate=250,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
):
    if high_cut >= srate / 2:
        raise ValueError("high_cut must be lower than the Nyquist frequency.")

    harmonic_filterbank = []

    for k in range(1, n_subbands + 1):
        low_cut = low_step * k

        if low_cut >= high_cut:
            raise ValueError("Invalid subband cutoff frequencies.")

        sos = butter(
            N=order,
            Wn=[low_cut, high_cut],
            btype="bandpass",
            fs=srate,
            output="sos",
        )

        harmonic_filterbank.append(sos)
        
    raw_weights = np.asarray(
        [(k ** (-1.25)) + 0.25 for k in range(1, n_subbands + 1)],
        dtype=float,
    )

    harmonic_filterweights = raw_weights / raw_weights.sum()

    return harmonic_filterbank, harmonic_filterweights


harmonic_filterbank, harmonic_filterweights = make_harmonic_filterbank(
    srate=sfreq,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
)

print("Number of harmonic subbands:", len(harmonic_filterbank))
print("Harmonic filter weights:", harmonic_filterweights)

def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.0


# ============================================================
# 3) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 Benchmark 数据；Bench_paradigm 和 Bench_dataset 需已在 Notebook 中定义。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "benchmark":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []

    for sid in selected_subjects:
        X_s, y_s, _ = Bench_paradigm.get_data(
            Bench_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return subjects_data, subjects_label, subject_ids, freqs_float


# ============================================================
# 4) One target subject
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
):
    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    X_cal, y_cal, X_test, y_test = subject_split_trials(
        X_target,
        y_target,
        n_calib=n_calib,
        rng=rng,
    )

    if len(y_test) == 0:
        return {
            "summary": [],
            "predictions": [],
        }

    source_X = []
    source_y = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue

        sid = subject_ids[src_idx]

        source_X.append(subjects_data[src_idx])
        source_y.append(subjects_label[src_idx])
        source_subjects.append(
            np.full(
                len(subjects_label[src_idx]),
                sid,
                dtype=int,
            )
        )

    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
    )

    calibration_X = X_cal if n_calib > 0 else None

    y_pred = model.predict(
        X_test,
        calib_X=calibration_X,
    )

    # ========================================================
    # 物理频率排序
    # ========================================================
    classes = [
        int(value)
        for value in model.classes_
    ]

    frequencies = [
        float(value)
        for value in model.freqs
    ]

    if len(classes) != len(frequencies):
        raise ValueError(
            "model.classes_ and model.freqs are not aligned."
        )

    frequency_by_class = dict(
        zip(classes, frequencies)
    )

    ordered_classes = sorted(
        classes,
        key=lambda cls: frequency_by_class[cls],
    )

    physical_rank = {
        cls: rank
        for rank, cls in enumerate(ordered_classes)
    }

    # ========================================================
    # Trial-level predictions
    # ========================================================
    prediction_rows = []

    correct = 0
    adjacent_errors = 0
    nonadjacent_errors = 0

    for trial_position, (
        true_class,
        predicted_class,
    ) in enumerate(zip(y_test, y_pred)):

        true_class = int(true_class)
        predicted_class = int(predicted_class)

        is_error = int(true_class != predicted_class)

        error_distance = abs(
            physical_rank[predicted_class]
            - physical_rank[true_class]
        )

        is_adjacent_error = int(
            is_error == 1
            and error_distance == 1
        )

        is_nonadjacent_error = int(
            is_error == 1
            and error_distance > 1
        )

        correct += int(is_error == 0)
        adjacent_errors += is_adjacent_error
        nonadjacent_errors += is_nonadjacent_error

        prediction_rows.append({
            "dataset": run_cfg.dataset_name,
            "window_sec": float(run_cfg.window_sec),
            "variant": variant.name,
            "target_subject_id": target_subject_id,
            "n_calib": int(n_calib),
            "seed": int(seed),
            "trial_position": int(trial_position),
            "true_class": true_class,
            "predicted_class": predicted_class,
            "true_frequency_hz": frequency_by_class[true_class],
            "predicted_frequency_hz": frequency_by_class[predicted_class],
            "true_physical_rank": physical_rank[true_class],
            "predicted_physical_rank": physical_rank[predicted_class],
            "error_distance": int(error_distance),
            "is_error": is_error,
            "is_adjacent_error": is_adjacent_error,
            "is_nonadjacent_error": is_nonadjacent_error,
        })

    n_test = len(y_test)
    total_errors = adjacent_errors + nonadjacent_errors
    acc = correct / n_test

    summary_row = {
        "dataset": run_cfg.dataset_name,
        "window_sec": float(run_cfg.window_sec),
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "acc": float(acc),
        "itr": compute_itr(
            acc,
            n_classes=run_cfg.n_classes,
            t_sec=run_cfg.decision_time,
        ),
        "n_test": int(n_test),
        "adjacent_error_rate": (
            adjacent_errors / n_test
        ),
        "nonadjacent_error_rate": (
            nonadjacent_errors / n_test
        ),
        "adjacent_error_share": (
            adjacent_errors / total_errors
            if total_errors > 0
            else 0.0
        ),
        "stc_split_mode": model.stc_split_mode,
        "enable_stage1": bool(model.enable_stage1),
        "enable_harmonic_branch": bool(
            model.enable_harmonic_branch
        ),
        "harmonic_filterbank_enabled": bool(
            model.harmonic_filterbank is not None
        ),
        "source_score_mode": model.source_score_mode,
        "confidence_lambda": float(
            model.confidence_lambda
        ),
    }

    return {
        "summary": [summary_row],
        "predictions": prediction_rows,
    }


# ============================================================
# 5) Parallel full LOSO
# ============================================================
def run_loso_variants_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):
    subjects_data, subjects_label, subject_ids, freqs_float = (
        load_dataset_for_loso_by_subjects(
            run_cfg.dataset_name,
            subjects,
        )
    )

    n_subj = len(subjects_data)

    summary_rows = []
    prediction_rows = []

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:

            seeds = (
                [0]
                if n_calib == 0
                else run_cfg.repeat_seeds
            )

            for seed in seeds:

                outputs = Parallel(
                    n_jobs=n_jobs,
                    backend=backend,
                    prefer=prefer,
                    verbose=verbose,
                )(
                    delayed(_run_one_target_task)(
                        tgt_idx=tgt_idx,
                        n_subj=n_subj,
                        subject_ids=subject_ids,
                        subjects_data=subjects_data,
                        subjects_label=subjects_label,
                        n_calib=n_calib,
                        seed=seed,
                        variant=variant,
                        base_model_kwargs=base_model_kwargs,
                        freqs_float=freqs_float,
                        run_cfg=run_cfg,
                    )
                    for tgt_idx in range(n_subj)
                )

                for output in outputs:
                    summary_rows.extend(
                        output["summary"]
                    )
                    prediction_rows.extend(
                        output["predictions"]
                    )

    return (
        pd.DataFrame(summary_rows),
        pd.DataFrame(prediction_rows),
    )


# ============================================================
# 6) Ablation variants
# ============================================================
variants = [
    # ========================================================
    # A. Stage 1/JFPM 实验
    # ========================================================
    VariantConfig("stage1_full",{"use_revised_method": True,"enable_stage1": True,"stc_split_mode": "time_ordered",},),
    VariantConfig("w/o_stage1",{"use_revised_method": True,"enable_stage1": False,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_radius0",{"use_revised_method": True,"enable_stage1": True,"neighbor_radius": 0,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_strength0",{"use_revised_method": True,"enable_stage1": True,"neighbor_strength": 0.0,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_strength0.5",{"use_revised_method": True,"enable_stage1": True,"neighbor_strength": 0.5,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_strength2",{"use_revised_method": True,"enable_stage1": True,"neighbor_strength": 2.0,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_radius2",{"use_revised_method": True,"enable_stage1": True,"neighbor_radius": 2,"stc_split_mode": "time_ordered",},),

    # ========================================================
    # B. confidence_lambda 敏感性
    # 重点观察 n_calib=2
    # ========================================================
    VariantConfig("lambda_0.0",{"confidence_lambda": 0.0,"stc_split_mode": "time_ordered",},),
    VariantConfig("lambda_0.1",{"confidence_lambda": 0.1,"stc_split_mode": "time_ordered",},),
    VariantConfig("lambda_0.2",{"confidence_lambda": 0.2,"stc_split_mode": "time_ordered",},),
    VariantConfig("lambda_0.5",{"confidence_lambda": 0.5,"stc_split_mode": "time_ordered",},),
    VariantConfig("lambda_1.0",{"confidence_lambda": 1.0,"stc_split_mode": "time_ordered",},),

    # ========================================================
    # C. Harmonic branch 实验
    # ========================================================
    VariantConfig("harmonic_raw",{"enable_harmonic_branch": True,"harmonic_filterbank": None,"harmonic_filterweights": None,"stc_split_mode": "time_ordered",},),
    VariantConfig("harmonic_filterbank",{"enable_harmonic_branch": True,"harmonic_filterbank": harmonic_filterbank,"harmonic_filterweights": harmonic_filterweights,"stc_split_mode": "time_ordered",},),
    VariantConfig("w/o_harmonic",{"enable_harmonic_branch": False,"stc_split_mode": "time_ordered",},),
    # ========================================================
    # D. STC split 敏感性
    # ========================================================
    VariantConfig("stc_time_ordered",{"stc_split_mode": "time_ordered",},),
    VariantConfig("stc_random",{"stc_split_mode": "random",},),
]


# ============================================================
# 7) Run
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.0

base_model_kwargs = dict(
    # 修订版 Stage 1
    use_revised_method=True,
    enable_stage1=True,

    neighbor_radius=1,
    neighbor_decay=0.5,
    neighbor_strength=1.0,

    # 修订版 source score
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,

    # calibration-only Stage 2
    target_alignment_mode="calibration",
    enable_stage2=True,

    # 主方法默认开启 harmonic branch
    enable_harmonic_branch=True,

    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)


run_cfg = RunConfig(
    dataset_name="Benchmark",
    n_calib_list=[0, 1, 2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
)

df_raw, df_predictions = run_loso_variants_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError("df_raw is empty.")

if df_predictions.empty:
    raise RuntimeError("df_predictions is empty.")


# ============================================================
# 8) Summary
# ============================================================

df_summary = (
    df_raw.groupby(
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
        ],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        adjacent_error_rate_mean=(
            "adjacent_error_rate",
            "mean",
        ),
        nonadjacent_error_rate_mean=(
            "nonadjacent_error_rate",
            "mean",
        ),
        n_runs=("acc", "count"),
    )
    .sort_values(
        [
            "variant",
            "n_calib",
        ]
    )
)


# ============================================================
# 9) Paired Wilcoxon + 95% CI
# ============================================================

def paired_comparison(
    df,
    left_variant,
    right_variant,
):
    key_cols = [
        "dataset",
        "window_sec",
        "target_subject_id",
        "n_calib",
        "seed",
    ]

    left = df[
        df["variant"] == left_variant
    ][key_cols + ["acc"]].rename(
        columns={"acc": "acc_left"}
    )

    right = df[
        df["variant"] == right_variant
    ][key_cols + ["acc"]].rename(
        columns={"acc": "acc_right"}
    )

    merged = left.merge(
        right,
        on=key_cols,
        how="inner",
    )

    rows = []

    for (
        dataset,
        window,
        n_calib,
    ), group in merged.groupby(
        [
            "dataset",
            "window_sec",
            "n_calib",
        ]
    ):
        delta = (
            group["acc_left"].to_numpy()
            - group["acc_right"].to_numpy()
        )

        delta = delta[np.isfinite(delta)]

        if len(delta) < 2:
            continue

        try:
            p_value = float(
                wilcoxon(
                    delta,
                    alternative="two-sided",
                ).pvalue
            )
        except ValueError:
            p_value = 1.0

        mean_delta = float(np.mean(delta))
        std_delta = float(np.std(delta, ddof=1))

        critical = float(
            t.ppf(
                0.975,
                df=len(delta) - 1,
            )
        )

        ci_half = (
            critical
            * std_delta
            / np.sqrt(len(delta))
        )

        rows.append({
            "comparison": (
                f"{left_variant} - {right_variant}"
            ),
            "dataset": dataset,
            "window_sec": float(window),
            "n_calib": int(n_calib),
            "n_pairs": int(len(delta)),
            "mean_delta_acc": mean_delta,
            "ci95_low": mean_delta - ci_half,
            "ci95_high": mean_delta + ci_half,
            "p_wilcoxon": p_value,
        })

    return rows


comparison_pairs = [
    ("stage1_full", "w/o_stage1"),
    ("harmonic_filterbank", "harmonic_raw"),
    ("harmonic_raw", "w/o_harmonic"),
    ("stc_random", "stc_time_ordered"),
    ("lambda_0.2", "lambda_0.0"),
]

available_variants = set(
    df_raw["variant"].unique()
)

paired_rows = []

for left_variant, right_variant in comparison_pairs:
    if {
        left_variant,
        right_variant,
    }.issubset(available_variants):

        paired_rows.extend(
            paired_comparison(
                df_raw,
                left_variant,
                right_variant,
            )
        )

df_paired = pd.DataFrame(paired_rows)


def holm_adjust(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)

    running = 0.0
    m = len(p_values)

    for rank, index in enumerate(order):
        value = min(
            1.0,
            p_values[index] * (m - rank),
        )
        running = max(running, value)
        adjusted[index] = running

    return adjusted


if not df_paired.empty:
    df_paired["p_holm"] = holm_adjust(
        df_paired["p_wilcoxon"].to_numpy()
    )


# ============================================================
# 10) Adjacent-frequency error rate
# ============================================================

def adjacent_summary(group):
    n_trials = len(group)
    n_errors = int(group["is_error"].sum())
    n_adjacent = int(
        group["is_adjacent_error"].sum()
    )
    n_nonadjacent = int(
        group["is_nonadjacent_error"].sum()
    )

    return pd.Series({
        "n_trials": n_trials,
        "accuracy": 1.0 - n_errors / n_trials,
        "adjacent_error_rate": (
            n_adjacent / n_trials
        ),
        "nonadjacent_error_rate": (
            n_nonadjacent / n_trials
        ),
        "adjacent_error_share": (
            n_adjacent / n_errors
            if n_errors > 0
            else 0.0
        ),
    })


df_adjacent = (
    df_predictions
    .groupby(
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
        ]
    )
    .apply(adjacent_summary)
    .reset_index()
)


# ============================================================
# 11) Complete confusion matrices
# ============================================================

confusion_rows = []

for group_keys, group in df_predictions.groupby(
    [
        "dataset",
        "window_sec",
        "variant",
        "n_calib",
    ]
):
    (
        dataset,
        window,
        variant,
        n_calib,
    ) = group_keys

    classes = sorted(
        set(group["true_class"])
        | set(group["predicted_class"])
    )

    counts = (
        group
        .groupby(
            [
                "true_class",
                "predicted_class",
            ]
        )
        .size()
        .to_dict()
    )

    for true_class in classes:
        for predicted_class in classes:
            confusion_rows.append({
                "dataset": dataset,
                "window_sec": float(window),
                "variant": variant,
                "n_calib": int(n_calib),
                "true_class": int(true_class),
                "predicted_class": int(
                    predicted_class
                ),
                "count": int(
                    counts.get(
                        (
                            true_class,
                            predicted_class,
                        ),
                        0,
                    )
                ),
            })


df_confusion = pd.DataFrame(confusion_rows)


# Wide-format confusion matrix for convenient plotting
df_confusion_wide = (
    df_confusion
    .pivot_table(
        index=[
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
            "true_class",
        ],
        columns="predicted_class",
        values="count",
        fill_value=0,
    )
    .reset_index()
)


# ============================================================
# 12) Save all detailed results
# ============================================================

df_raw.to_csv(
    f"subject_accuracy_{window_sec:.1f}s.csv",
    index=False,
)

df_summary.to_csv(
    f"summary_{window_sec:.1f}s.csv",
    index=False,
)

df_predictions.to_csv(
    f"trial_predictions_{window_sec:.1f}s.csv",
    index=False,
)

df_paired.to_csv(
    f"paired_wilcoxon_ci_holm_{window_sec:.1f}s.csv",
    index=False,
)

df_adjacent.to_csv(
    f"adjacent_frequency_errors_{window_sec:.1f}s.csv",
    index=False,
)

df_confusion.to_csv(
    f"confusion_matrices_long_{window_sec:.1f}s.csv",
    index=False,
)

df_confusion_wide.to_csv(
    f"confusion_matrices_wide_{window_sec:.1f}s.csv",
    index=False,
)


# ============================================================
# 13) Concise printing
# ============================================================

print("\n========== Key Accuracy Results ==========")

key_variants = [
    "stage1_full",
    "w/o_stage1",
    "harmonic_raw",
    "harmonic_filterbank",
    "w/o_harmonic",
    "stc_time_ordered",
    "stc_random",
    "lambda_0.0",
    "lambda_0.2",
    "lambda_1.0",
]

print(
    df_summary[
        df_summary["variant"].isin(key_variants)
    ][
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
            "acc_mean",
            "itr_mean",
        ]
    ].to_string(index=False)
)

print("\n========== Paired Statistics: n_calib=2 ==========")

if not df_paired.empty:
    print(
        df_paired[
            df_paired["n_calib"] == 2
        ][
            [
                "comparison",
                "dataset",
                "window_sec",
                "mean_delta_acc",
                "ci95_low",
                "ci95_high",
                "p_wilcoxon",
                "p_holm",
            ]
        ].to_string(index=False)
    )

print("\n========== Adjacent-Frequency Errors: n_calib=2 ==========")

print(
    df_adjacent[
        df_adjacent["n_calib"] == 2
    ].to_string(index=False)
)
print("\nDetailed CSV files saved.")


Number of harmonic subbands: 5
Harmonic filter weights: [0.38648818 0.20729625 0.15560897 0.13195532 0.11865127]
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.m

In [3]:
# ============================================================
# 1) Utilities
# ============================================================
def make_harmonic_filterbank(
    srate=250,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
):
    if high_cut >= srate / 2:
        raise ValueError("high_cut must be lower than the Nyquist frequency.")

    harmonic_filterbank = []

    for k in range(1, n_subbands + 1):
        low_cut = low_step * k

        if low_cut >= high_cut:
            raise ValueError("Invalid subband cutoff frequencies.")

        sos = butter(
            N=order,
            Wn=[low_cut, high_cut],
            btype="bandpass",
            fs=srate,
            output="sos",
        )

        harmonic_filterbank.append(sos)
        
    raw_weights = np.asarray(
        [(k ** (-1.25)) + 0.25 for k in range(1, n_subbands + 1)],
        dtype=float,
    )

    harmonic_filterweights = raw_weights / raw_weights.sum()

    return harmonic_filterbank, harmonic_filterweights


harmonic_filterbank, harmonic_filterweights = make_harmonic_filterbank(
    srate=sfreq,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
)

print("Number of harmonic subbands:", len(harmonic_filterbank))
print("Harmonic filter weights:", harmonic_filterweights)

def subject_split_indices(y_subj, n_calib, rng):
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    return (
        np.asarray(calib_idx, dtype=int),
        np.asarray(test_idx, dtype=int),
    )


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.0


# ============================================================
# 3) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 Benchmark 数据及与 trial 顺序对齐的真实 session/run block 标签。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "benchmark":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []
    block_ids_by_subject = {}

    for sid in selected_subjects:
        X_s, y_s, meta_s = Bench_paradigm.get_data(
            Bench_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        if not isinstance(meta_s, pd.DataFrame):
            raise TypeError(
                f"Subject {sid}: expected pandas DataFrame metadata, "
                f"got {type(meta_s).__name__}."
            )
        if len(meta_s) != len(y_s):
            raise ValueError(
                f"Subject {sid}: metadata/trial length mismatch: "
                f"{len(meta_s)} != {len(y_s)}."
            )
        if not {"session", "run"}.issubset(meta_s.columns):
            raise ValueError(
                f"Subject {sid}: metadata must contain session and run. "
                f"Available columns: {meta_s.columns.tolist()}"
            )
        if meta_s[["session", "run"]].isna().any().any():
            raise ValueError(
                f"Subject {sid}: session/run metadata contains missing values."
            )

        # meta_s is returned in the same epoch order as X_s/y_s.  The pair
        # (session, run) therefore gives one real acquisition block label per
        # trial, without inferring any block from trial indices.
        block_ids_by_subject[sid] = np.asarray(
            [
                f"subject={sid}|session={session}|run={run}"
                for session, run in zip(meta_s["session"], meta_s["run"])
            ],
            dtype=object,
        )

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return (
        subjects_data,
        subjects_label,
        subject_ids,
        freqs_float,
        block_ids_by_subject,
    )


# ============================================================
# 4) One target subject
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
    block_ids_by_subject=None,
):

    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    calib_idx, test_idx = subject_split_indices(
        y_target,
        n_calib=n_calib,
        rng=rng,
    )
    target_subject_id = subject_ids[tgt_idx]

    target_block_ids = None
    if block_ids_by_subject is not None:
        if target_subject_id not in block_ids_by_subject:
            raise KeyError(
                f"Missing block IDs for target subject {target_subject_id}"
            )
    
        target_block_ids = np.asarray(
            block_ids_by_subject[target_subject_id],
            dtype=object,
        )
    
        if len(target_block_ids) != len(y_target):
            raise ValueError(
                f"Target subject {target_subject_id}: "
                "block_ids length does not match trial count."
            )
    
    target_calib_block_ids = (
        target_block_ids[calib_idx]
        if target_block_ids is not None and n_calib > 0
        else None
    )
    
    X_cal = X_target[calib_idx]
    y_cal = y_target[calib_idx]
    X_test = X_target[test_idx]
    y_test = y_target[test_idx]


    if len(y_test) == 0:
        return {
            "summary": [],
            "predictions": [],
        }

    source_X = []
    source_y = []
    source_block_ids_list = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue
    
        sid = subject_ids[src_idx]
        X_source_subject = subjects_data[src_idx]
        y_source_subject = subjects_label[src_idx]
    
        source_X.append(X_source_subject)
        source_y.append(y_source_subject)
        source_subjects.append(
            np.full(len(y_source_subject), sid, dtype=int)
        )
    
        if block_ids_by_subject is not None:
            if sid not in block_ids_by_subject:
                raise KeyError(f"Missing block IDs for source subject {sid}")
    
            sid_block_ids = np.asarray(
                block_ids_by_subject[sid],
                dtype=object,
            )
    
            if len(sid_block_ids) != len(y_source_subject):
                raise ValueError(
                    f"Source subject {sid}: "
                    "block_ids length does not match trial count."
                )
    
            source_block_ids_list.append(sid_block_ids)


    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)
    source_block_ids = (
        np.concatenate(source_block_ids_list, axis=0)
        if source_block_ids_list
        else None
    )

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    if (model_kwargs.get("stc_split_mode") == "block_stratified"
        and source_block_ids is None
    ):
        raise ValueError(
            "block_stratified requires real source block/run labels."
        )
    
    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
        source_block_ids=source_block_ids,
        target_calib_block_ids=target_calib_block_ids,
    )


    calibration_X = X_cal if n_calib > 0 else None

    y_pred = model.predict(
        X_test,
        calib_X=calibration_X,
    )

    # ========================================================
    # 物理频率排序
    # ========================================================
    classes = [
        int(value)
        for value in model.classes_
    ]

    frequencies = [
        float(value)
        for value in model.freqs
    ]

    if len(classes) != len(frequencies):
        raise ValueError(
            "model.classes_ and model.freqs are not aligned."
        )

    frequency_by_class = dict(
        zip(classes, frequencies)
    )

    ordered_classes = sorted(
        classes,
        key=lambda cls: frequency_by_class[cls],
    )

    physical_rank = {
        cls: rank
        for rank, cls in enumerate(ordered_classes)
    }

    # ========================================================
    # Trial-level predictions
    # ========================================================
    prediction_rows = []

    correct = 0
    adjacent_errors = 0
    nonadjacent_errors = 0

    for trial_position, (
        true_class,
        predicted_class,
    ) in enumerate(zip(y_test, y_pred)):

        true_class = int(true_class)
        predicted_class = int(predicted_class)

        is_error = int(true_class != predicted_class)

        error_distance = abs(
            physical_rank[predicted_class]
            - physical_rank[true_class]
        )

        is_adjacent_error = int(
            is_error == 1
            and error_distance == 1
        )

        is_nonadjacent_error = int(
            is_error == 1
            and error_distance > 1
        )

        correct += int(is_error == 0)
        adjacent_errors += is_adjacent_error
        nonadjacent_errors += is_nonadjacent_error

        prediction_rows.append({
            "dataset": run_cfg.dataset_name,
            "window_sec": float(run_cfg.window_sec),
            "variant": variant.name,
            "target_subject_id": target_subject_id,
            "n_calib": int(n_calib),
            "seed": int(seed),
            "trial_position": int(trial_position),
            "true_class": true_class,
            "predicted_class": predicted_class,
            "true_frequency_hz": frequency_by_class[true_class],
            "predicted_frequency_hz": frequency_by_class[predicted_class],
            "true_physical_rank": physical_rank[true_class],
            "predicted_physical_rank": physical_rank[predicted_class],
            "error_distance": int(error_distance),
            "is_error": is_error,
            "is_adjacent_error": is_adjacent_error,
            "is_nonadjacent_error": is_nonadjacent_error,
        })

    n_test = len(y_test)
    total_errors = adjacent_errors + nonadjacent_errors
    acc = correct / n_test

    summary_row = {
        "dataset": run_cfg.dataset_name,
        "window_sec": float(run_cfg.window_sec),
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "acc": float(acc),
        "itr": compute_itr(
            acc,
            n_classes=run_cfg.n_classes,
            t_sec=run_cfg.decision_time,
        ),
        "n_test": int(n_test),
        "adjacent_error_rate": (
            adjacent_errors / n_test
        ),
        "nonadjacent_error_rate": (
            nonadjacent_errors / n_test
        ),
        "adjacent_error_share": (
            adjacent_errors / total_errors
            if total_errors > 0
            else 0.0
        ),
        "stc_split_mode": model.stc_split_mode,
        "enable_stage1": bool(model.enable_stage1),
        "enable_harmonic_branch": bool(
            model.enable_harmonic_branch
        ),
        "harmonic_filterbank_enabled": bool(
            model.harmonic_filterbank is not None
        ),
        "source_score_mode": model.source_score_mode,
        "confidence_lambda": float(
            model.confidence_lambda
        ),
    }

    return {
        "summary": [summary_row],
        "predictions": prediction_rows,
    }


# ============================================================
# 5) Parallel full LOSO
# ============================================================
def run_loso_variants_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    block_ids_by_subject=None,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):

    (
        subjects_data,
        subjects_label,
        subject_ids,
        freqs_float,
        loaded_block_ids_by_subject,
    ) = (
        load_dataset_for_loso_by_subjects(
            run_cfg.dataset_name,
            subjects,
        )
    )

    # 默认使用 loader 从真实 metadata 提取的 (session, run) 标签；
    # 仅在调用者显式传入时才使用外部映射。
    if block_ids_by_subject is None:
        block_ids_by_subject = loaded_block_ids_by_subject

    block_count_by_subject = {
        sid: int(len(np.unique(block_ids_by_subject[sid])))
        for sid in subject_ids
    }
    print(
        "[block_stratified] real session/run blocks per subject: "
        f"min={min(block_count_by_subject.values())}, "
        f"max={max(block_count_by_subject.values())}"
    )

    n_subj = len(subjects_data)

    summary_rows = []
    prediction_rows = []

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:

            seeds = (
                [0]
                if n_calib == 0
                else run_cfg.repeat_seeds
            )

            for seed in seeds:

                outputs = Parallel(
                    n_jobs=n_jobs,
                    backend=backend,
                    prefer=prefer,
                    verbose=verbose,
                )(
                    delayed(_run_one_target_task)(
                         tgt_idx=tgt_idx,
                        n_subj=n_subj,
                        subject_ids=subject_ids,
                        subjects_data=subjects_data,
                        subjects_label=subjects_label,
                        n_calib=n_calib,
                        seed=seed,
                        variant=variant,
                        base_model_kwargs=base_model_kwargs,
                        freqs_float=freqs_float,
                        run_cfg=run_cfg,
                        block_ids_by_subject=block_ids_by_subject,
                    )
                    for tgt_idx in range(n_subj)
                )

                for output in outputs:
                    summary_rows.extend(
                        output["summary"]
                    )
                    prediction_rows.extend(
                        output["predictions"]
                    )

    return (
        pd.DataFrame(summary_rows),
        pd.DataFrame(prediction_rows),
    )


# ============================================================
# 6) STC split sensitivity only
# ============================================================
# 除 STC split 外，两个条件使用完全相同的修订版 SQ-HAF 设置。
# 不运行 Stage 1、lambda 或 harmonic branch 消融。
variants = [
    VariantConfig(
        "stc_time_ordered",
        {
            "use_revised_method": True,
            "enable_stage1": True,
            "stc_split_mode": "time_ordered",
        },
    ),
    VariantConfig(
        "stc_block_stratified",
        {
            "use_revised_method": True,
            "enable_stage1": True,
            "stc_split_mode": "block_stratified",
        },
    ),
]


# ============================================================
# 7) Run
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.0

base_model_kwargs = dict(
    # 修订版 Stage 1
    use_revised_method=True,
    enable_stage1=True,

    neighbor_radius=1,
    neighbor_decay=0.5,
    neighbor_strength=1.0,

    # 修订版 source score
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,

    # calibration-only Stage 2
    target_alignment_mode="calibration",
    enable_stage2=True,

    # 主协议；block-stratified 仅由对应 sensitivity variant 覆盖。
    stc_split_mode="time_ordered",

    # 主方法默认开启 harmonic branch
    enable_harmonic_branch=True,

    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)


run_cfg = RunConfig(
    dataset_name="Benchmark",
    n_calib_list=[0, 1, 2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
)

df_raw, df_predictions = run_loso_variants_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError("df_raw is empty.")

if df_predictions.empty:
    raise RuntimeError("df_predictions is empty.")

expected_split_modes = {"time_ordered", "block_stratified"}
observed_split_modes = set(
    df_raw["stc_split_mode"].dropna().astype(str).unique()
)
if observed_split_modes != expected_split_modes:
    raise AssertionError(
        "STC sensitivity run must contain time_ordered and block_stratified only; "
        f"observed={observed_split_modes}"
    )


# ============================================================
# 8) Summary
# ============================================================

df_summary = (
    df_raw.groupby(
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
        ],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        adjacent_error_rate_mean=(
            "adjacent_error_rate",
            "mean",
        ),
        nonadjacent_error_rate_mean=(
            "nonadjacent_error_rate",
            "mean",
        ),
        n_runs=("acc", "count"),
    )
    .sort_values(
        [
            "variant",
            "n_calib",
        ]
    )
)


# ============================================================
# 9) Paired Wilcoxon + 95% CI
# ============================================================

def paired_comparison(df, left_variant, right_variant):
    """Paired block-minus-time statistics for ACC and ITR.

    n_calib=1 is retained in the saved table as a descriptive check only:
    the STC confidence gate is zero in that setting, so it is excluded from
    the planned inferential family and Holm correction.
    """
    key_cols = [
        "dataset",
        "window_sec",
        "target_subject_id",
        "n_calib",
        "seed",
    ]

    left = df[df["variant"] == left_variant][
        key_cols + ["acc", "itr"]
    ].rename(
        columns={
            "acc": "acc_left",
            "itr": "itr_left",
        }
    )

    right = df[df["variant"] == right_variant][
        key_cols + ["acc", "itr"]
    ].rename(
        columns={
            "acc": "acc_right",
            "itr": "itr_right",
        }
    )

    merged = left.merge(
        right,
        on=key_cols,
        how="inner",
    )

    rows = []

    for (
        dataset,
        window,
        n_calib,
    ), group in merged.groupby(
        [
            "dataset",
            "window_sec",
            "n_calib",
        ]
    ):
        for metric in ("acc", "itr"):
            delta = (
                group[f"{metric}_left"].to_numpy(dtype=float)
                - group[f"{metric}_right"].to_numpy(dtype=float)
            )
            delta = delta[np.isfinite(delta)]

            if len(delta) < 2:
                continue

            mean_delta = float(np.mean(delta))
            std_delta = float(np.std(delta, ddof=1))
            critical = float(t.ppf(0.975, df=len(delta) - 1))
            ci_half = critical * std_delta / np.sqrt(len(delta))

            inferential = int(n_calib) in {0, 2}
            if not inferential or np.allclose(delta, 0.0):
                p_value = 1.0 if inferential else np.nan
            else:
                try:
                    p_value = float(
                        wilcoxon(delta, alternative="two-sided").pvalue
                    )
                except ValueError:
                    p_value = 1.0

            rows.append({
                "comparison": f"{left_variant} - {right_variant}",
                "metric": metric,
                "dataset": dataset,
                "window_sec": float(window),
                "n_calib": int(n_calib),
                "n_pairs": int(len(delta)),
                "mean_delta": mean_delta,
                "ci95_low": mean_delta - ci_half,
                "ci95_high": mean_delta + ci_half,
                "descriptive_only": not inferential,
                "p_wilcoxon": p_value,
            })

    return rows


comparison_pairs = [
    ("stc_block_stratified", "stc_time_ordered"),
]

available_variants = set(
    df_raw["variant"].unique()
)

paired_rows = []

for left_variant, right_variant in comparison_pairs:
    if {
        left_variant,
        right_variant,
    }.issubset(available_variants):

        paired_rows.extend(
            paired_comparison(
                df_raw,
                left_variant,
                right_variant,
            )
        )

df_paired = pd.DataFrame(paired_rows)


def holm_adjust(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)

    running = 0.0
    m = len(p_values)

    for rank, index in enumerate(order):
        value = min(
            1.0,
            p_values[index] * (m - rank),
        )
        running = max(running, value)
        adjusted[index] = running

    return adjusted


if not df_paired.empty:
    df_paired["p_holm"] = np.nan
    inferential_rows = df_paired["p_wilcoxon"].notna()
    df_paired.loc[inferential_rows, "p_holm"] = holm_adjust(
        df_paired.loc[inferential_rows, "p_wilcoxon"].to_numpy()
    )


# ============================================================
# 10) Adjacent-frequency error rate
# ============================================================

def adjacent_summary(group):
    n_trials = len(group)
    n_errors = int(group["is_error"].sum())
    n_adjacent = int(
        group["is_adjacent_error"].sum()
    )
    n_nonadjacent = int(
        group["is_nonadjacent_error"].sum()
    )

    return pd.Series({
        "n_trials": n_trials,
        "accuracy": 1.0 - n_errors / n_trials,
        "adjacent_error_rate": (
            n_adjacent / n_trials
        ),
        "nonadjacent_error_rate": (
            n_nonadjacent / n_trials
        ),
        "adjacent_error_share": (
            n_adjacent / n_errors
            if n_errors > 0
            else 0.0
        ),
    })


df_adjacent = (
    df_predictions
    .groupby(
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
        ]
    )
    .apply(adjacent_summary)
    .reset_index()
)


# ============================================================
# 11) Complete confusion matrices
# ============================================================

confusion_rows = []

for group_keys, group in df_predictions.groupby(
    [
        "dataset",
        "window_sec",
        "variant",
        "n_calib",
    ]
):
    (
        dataset,
        window,
        variant,
        n_calib,
    ) = group_keys

    classes = sorted(
        set(group["true_class"])
        | set(group["predicted_class"])
    )

    counts = (
        group
        .groupby(
            [
                "true_class",
                "predicted_class",
            ]
        )
        .size()
        .to_dict()
    )

    for true_class in classes:
        for predicted_class in classes:
            confusion_rows.append({
                "dataset": dataset,
                "window_sec": float(window),
                "variant": variant,
                "n_calib": int(n_calib),
                "true_class": int(true_class),
                "predicted_class": int(
                    predicted_class
                ),
                "count": int(
                    counts.get(
                        (
                            true_class,
                            predicted_class,
                        ),
                        0,
                    )
                ),
            })


df_confusion = pd.DataFrame(confusion_rows)


# Wide-format confusion matrix for convenient plotting
df_confusion_wide = (
    df_confusion
    .pivot_table(
        index=[
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
            "true_class",
        ],
        columns="predicted_class",
        values="count",
        fill_value=0,
    )
    .reset_index()
)


# ============================================================
# 12) Save all detailed results
# ============================================================

df_raw.to_csv(
    f"stc_sensitivity_subject_results_{window_sec:.1f}s.csv",
    index=False,
)

df_summary.to_csv(
    f"stc_sensitivity_summary_{window_sec:.1f}s.csv",
    index=False,
)

df_predictions.to_csv(
    f"stc_sensitivity_trial_predictions_{window_sec:.1f}s.csv",
    index=False,
)

df_paired.to_csv(
    f"stc_sensitivity_paired_wilcoxon_ci_holm_{window_sec:.1f}s.csv",
    index=False,
)

df_adjacent.to_csv(
    f"stc_sensitivity_adjacent_frequency_errors_{window_sec:.1f}s.csv",
    index=False,
)

df_confusion.to_csv(
    f"stc_sensitivity_confusion_matrices_long_{window_sec:.1f}s.csv",
    index=False,
)

df_confusion_wide.to_csv(
    f"stc_sensitivity_confusion_matrices_wide_{window_sec:.1f}s.csv",
    index=False,
)


# ============================================================
# 13) Concise printing
# ============================================================

print("\n========== STC Split Sensitivity ==========")

key_variants = [
    "stc_time_ordered",
    "stc_block_stratified",
]

print(
    df_summary[
        df_summary["variant"].isin(key_variants)
    ][
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
            "acc_mean",
            "itr_mean",
        ]
    ].to_string(index=False)
)

print("\n========== Paired Block-Stratified − Time-Ordered ==========")

if not df_paired.empty:
    print(
        df_paired[
            df_paired["n_calib"].isin([0, 2])
        ][
            [
                "comparison",
                "metric",
                "dataset",
                "window_sec",
                "n_calib",
                "n_pairs",
                "mean_delta",
                "ci95_low",
                "ci95_high",
                "p_wilcoxon",
                "p_holm",
            ]
        ].to_string(index=False)
    )

print("\n========== n_calib=1 Descriptive Check ==========")

if not df_paired.empty:
    print(
        df_paired[
            df_paired["n_calib"] == 1
        ][
            [
                "comparison",
                "metric",
                "dataset",
                "window_sec",
                "n_pairs",
                "mean_delta",
                "ci95_low",
                "ci95_high",
                "descriptive_only",
            ]
        ].to_string(index=False)
    )

print("\nSTC sensitivity CSV files saved.")

Number of harmonic subbands: 5
Harmonic filter weights: [0.38648818 0.20729625 0.15560897 0.13195532 0.11865127]
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.m

In [3]:

# ============================================================
# 1) Utilities
# ============================================================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Five-subband harmonic filter bank
# ============================================================
def make_harmonic_filterbank(
    srate=250,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
):
    """
    Build the five-subband filter bank used only by the
    harmonic-reference branch.

    Subbands:
        1:  6-90 Hz
        2: 12-90 Hz
        3: 18-90 Hz
        4: 24-90 Hz
        5: 30-90 Hz

    Normalized weights:
        w_k proportional to k^(-1.25) + 0.25
    """
    if high_cut >= srate / 2:
        raise ValueError("high_cut must be lower than the Nyquist frequency.")

    harmonic_filterbank = []

    for k in range(1, n_subbands + 1):
        low_cut = low_step * k

        if low_cut >= high_cut:
            raise ValueError("Invalid subband cutoff frequencies.")

        sos = butter(
            N=order,
            Wn=[low_cut, high_cut],
            btype="bandpass",
            fs=srate,
            output="sos",
        )
        harmonic_filterbank.append(sos)

    raw_weights = np.asarray(
        [(k ** (-1.25)) + 0.25 for k in range(1, n_subbands + 1)],
        dtype=float,
    )
    harmonic_filterweights = raw_weights / raw_weights.sum()

    return harmonic_filterbank, harmonic_filterweights


# ============================================================
# 3) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.0


# ============================================================
# 4) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 Benchmark 数据；Bench_paradigm 和 Bench_dataset 需已在 Notebook 中定义。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "benchmark":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []

    for sid in selected_subjects:
        X_s, y_s, _ = Bench_paradigm.get_data(
            Bench_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return subjects_data, subjects_label, subject_ids, freqs_float


# ============================================================
# 5) One target subject
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
):
    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    X_cal, y_cal, X_test, y_test = subject_split_trials(
        X_target,
        y_target,
        n_calib=n_calib,
        rng=rng,
    )

    if len(y_test) == 0:
        return []

    source_X = []
    source_y = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue

        sid = subject_ids[src_idx]
        X_source_subject = subjects_data[src_idx]
        y_source_subject = subjects_label[src_idx]

        source_X.append(X_source_subject)
        source_y.append(y_source_subject)
        source_subjects.append(
            np.full(len(y_source_subject), sid, dtype=int)
        )

    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)

    # 使 target split、STC random split 和 source random selection 使用相同 seed
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
    )

    calibration_X = X_cal if n_calib > 0 else None
    y_pred = model.predict(X_test, calib_X=calibration_X)

    acc = float(np.mean(y_pred == y_test))
    itr = compute_itr(
        acc,
        n_classes=run_cfg.n_classes,
        t_sec=run_cfg.decision_time,
    )

    return [{
        "dataset": run_cfg.dataset_name,
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "acc": acc,
        "itr": itr,
        "n_test": int(len(y_test)),
        "stc_split_mode": model.stc_split_mode,
        "enable_harmonic_branch": bool(model.enable_harmonic_branch),
        "harmonic_n_subbands": (
            0
            if not model.enable_harmonic_branch
            else (
                len(model.harmonic_filterbank)
                if model.harmonic_filterbank is not None
                else 1
            )
        ),
        "enable_stage1": bool(model.enable_stage1),
        "enable_stage2": bool(model.enable_stage2),
        "enable_branch_r2": bool(model.enable_branch_r2),
        "enable_branch_r3": bool(model.enable_branch_r3),
    }]


# ============================================================
# 6) Parallel full LOSO
# ============================================================
def run_loso_variants_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):
    subjects_data, subjects_label, subject_ids, freqs_float = (
        load_dataset_for_loso_by_subjects(
            run_cfg.dataset_name,
            subjects,
        )
    )

    n_subj = len(subjects_data)
    rows = []

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:
            seeds = [0] if n_calib == 0 else run_cfg.repeat_seeds

            for seed in seeds:
                out = Parallel(
                    n_jobs=n_jobs,
                    backend=backend,
                    prefer=prefer,
                    verbose=verbose,
                )(
                    delayed(_run_one_target_task)(
                        tgt_idx=tgt_idx,
                        n_subj=n_subj,
                        subject_ids=subject_ids,
                        subjects_data=subjects_data,
                        subjects_label=subjects_label,
                        n_calib=n_calib,
                        seed=seed,
                        variant=variant,
                        base_model_kwargs=base_model_kwargs,
                        freqs_float=freqs_float,
                        run_cfg=run_cfg,
                    )
                    for tgt_idx in range(n_subj)
                )

                for result in out:
                    if result:
                        rows.extend(result)

    return pd.DataFrame(rows)


# ============================================================
# 7) Ablation variants
# ============================================================
# Every variant below inherits the five-subband harmonic-reference
# baseline from base_model_kwargs. Each ablation changes only the
# explicitly listed parameter(s).
variants = [
    # =====================================================
    # Main method: five-subband harmonic-reference baseline
    # =====================================================
    VariantConfig("full",{
            "source_score_mode": "similarity_confidence",
            "stc_split_mode": "time_ordered",
            "enable_harmonic_branch": True,},),
    # =====================================================
    # Source-selection ablation
    # =====================================================

    # 随机选择源被试
    VariantConfig("random_source_selection",{"source_score_mode": "random","source_weight_mode": "uniform",},),
    # 只按目标-源模板相似度评分，不使用 confidence penalty
    VariantConfig("similarity_only_source_scoring",{"source_score_mode": "similarity",},),
    # 只按源被试稳定性评分
    VariantConfig("stability_only_source_scoring",{"source_score_mode": "stability",},),
    # 保留完整 source ranking，但所有选中源使用相同权重
    VariantConfig("uniform_source_weighting",{"source_weight_mode": "uniform",},),
    #  similarity × STC 方法
    VariantConfig("legacy_similarity_stc",{"source_score_mode": "legacy_multiplicative",},),
    # =====================================================
    # Alignment ablation
    # =====================================================
    # 当前实现中 Stage 2 对应 global covariance alignment
    VariantConfig("w/o_global_alignment",{"enable_stage2": False,},),
    # 当前实现中 Stage 1 对应 hierarchical/local alignment
    VariantConfig("w/o_hierarchical_local_alignment",{"enable_stage1": False,},),
    # =====================================================
    # Branch ablation
    # =====================================================
    # harmonic-reference branch，即 r1
    VariantConfig("w/o_harmonic_reference_branch",{"enable_harmonic_branch": False,},),
    # generalized source-template branch，即 r2
    VariantConfig("w/o_source_template_branch",{"enable_branch_r2": False,},),
    # source-specific template branch，即 r3
    VariantConfig("w/o_source_specific_template_branch",{"enable_branch_r3": False,},),
    # =====================================================
    # Fusion ablation
    # =====================================================
    VariantConfig("absolute_value_fusion",{"fusion_mode": "abs_sum",},),
    # =====================================================
    # Reviewer 4: STC split sensitivity
    # =====================================================
    VariantConfig("stc_random",{"stc_split_mode": "random",},),
]


# ============================================================
# 8) Run
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.0

# ------------------------------------------------------------
# Build the five-subband harmonic-reference baseline.
# IMPORTANT:
#   Use harmonic_filterbank / harmonic_filterweights here.
#   Do NOT assign these filters to the model's main filterbank,
#   because the goal is to change only the harmonic-reference branch.
# ------------------------------------------------------------
harmonic_filterbank, harmonic_filterweights = make_harmonic_filterbank(
    srate=sfreq,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
)

print("Number of harmonic subbands:", len(harmonic_filterbank))
print("Harmonic filter weights:", harmonic_filterweights)

base_model_kwargs = dict(
    # Revised SQ-HAF
    use_revised_method=True,

    # Source selection
    n_sources=10,
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,

    # Stage 1
    neighbor_radius=1,
    neighbor_decay=0.5,
    neighbor_strength=1.0,
    enable_stage1=True,

    # Stage 2
    target_alignment_mode="calibration",
    enable_stage2=True,

    # STC
    stc_split_mode="time_ordered",

    # Three scoring branches
    enable_harmonic_branch=True,
    enable_branch_r2=True,
    enable_branch_r3=True,

    # --------------------------------------------------------
    # Five-subband harmonic-reference branch baseline
    # --------------------------------------------------------
    harmonic_filterbank=harmonic_filterbank,
    harmonic_filterweights=harmonic_filterweights,

    # Sinusoidal references
    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)

run_cfg = RunConfig(
    dataset_name="Benchmark",
    n_calib_list=[0, 1, 2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
)

df_raw = run_loso_variants_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError("df_raw is empty. Please check data loading / model fitting.")

# ============================================================
# Configuration audit
# ============================================================
# Except for the harmonic-branch ablation itself, every variant should
# report harmonic_n_subbands == 5. The harmonic-branch ablation should
# report harmonic_n_subbands == 0.
audit_cols = [
    "variant",
    "harmonic_n_subbands",
    "enable_stage1",
    "enable_stage2",
    "enable_harmonic_branch",
    "enable_branch_r2",
    "enable_branch_r3",
    "stc_split_mode",
]

df_audit = (
    df_raw[audit_cols]
    .drop_duplicates()
    .sort_values("variant")
)

print("\n========== Configuration Audit ==========")
print(df_audit.to_string(index=False))

unexpected = df_audit[
    (df_audit["enable_harmonic_branch"])
    & (df_audit["harmonic_n_subbands"] != 5)
]
if not unexpected.empty:
    raise RuntimeError(
        "Some harmonic-enabled variants are not using the five-subband "
        "harmonic baseline:\n"
        + unexpected.to_string(index=False)
    )

harmonic_off = df_audit[
    ~df_audit["enable_harmonic_branch"]
]
if not harmonic_off.empty and not (harmonic_off["harmonic_n_subbands"] == 0).all():
    raise RuntimeError(
        "The no-harmonic ablation has an unexpected harmonic_n_subbands value."
    )


# ============================================================
# 9) Compact final summary
# ============================================================
df_summary = (
    df_raw.groupby(
        ["dataset", "variant", "n_calib"],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        n_runs=("acc", "count"),
    )
    .sort_values(["variant", "n_calib"])
)

print("\n========== Ablation Summary ==========")
print(
    df_summary[
        [
            "dataset",
            "variant",
            "n_calib",
            "acc_mean",
            "acc_std",
            "itr_mean",
            "itr_std",
            "n_runs",
        ]
    ].to_string(index=False)
)

Number of harmonic subbands: 5
Harmonic filter weights: [0.38648818 0.20729625 0.15560897 0.13195532 0.11865127]
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.m

In [3]:
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """
    ITR bits/min
    acc: 准确率[0,1]
    n_classes: 类别数
    t_sec: 单次决策时间(秒)
    """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(dataset=dataset, subjects=[s], return_concat=True)
        X = np.asarray(X)
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


# =========================
# 全体被试 LOSO（不分组）
# =========================
all_subjects = list(range(1, 36))
n_classes = 40   # 按你的数据集类别数设置
T = 1.0          # 单次决策时间(秒)，按实验窗口修改

print("\n========== LOSO on ALL subjects ==========")
print(f"n_classes={n_classes}, T={T}s")

all_data = build_subject_data(Bench_paradigm, Bench_dataset, all_subjects)

all_results = []
accs, itrs = [], []

for target_sid in all_subjects:
    source_sids = [s for s in all_subjects if s != target_sid]
    source_data = {s: all_data[s] for s in source_sids}
    X_t, y_t = all_data[target_sid]

    model = SUTLSSVEP(
        srate=sfreq,
        freqs=[float(f) for f in freq_list],
        n_harmonics=3,
        n_bands=5,
        top_m1=10,
        n_jobs=8
    )

    model.fit(source_data)
    y_pred = model.predict(X_t)

    acc = float(np.mean(y_pred == y_t))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=T))

    accs.append(acc)
    itrs.append(itr)

    print(f"target S{target_sid}: acc={acc:.4f}, itr={itr:.2f} bits/min")

    all_results.append({
        "subject_id": target_sid,
        "subject": f"S{target_sid}",
        "acc": acc,
        "itr": itr,
        "n_trials": len(y_t)
    })

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


# =========================
# 最终汇总打印（仅打印，不保存csv）
# =========================
df = pd.DataFrame(all_results).sort_values("subject_id")

print("\n========== Subject-wise Results ==========")
print(df.to_string(index=False))

print("\n========== Overall Summary ==========")
summary_df = pd.DataFrame([{
    "acc_mean": df["acc"].mean(),
    "acc_std": df["acc"].std(),
    "itr_mean(bits/min)": df["itr"].mean(),
    "itr_std(bits/min)": df["itr"].std(),
    "n_subjects": df["subject_id"].nunique()
}])
print(summary_df.to_string(index=False))

print(f"\nOverall mean acc = {df['acc'].mean():.4f}")
print(f"Overall mean itr = {df['itr'].mean():.2f} bits/min")


========== LOSO on ALL subjects ==========
n_classes=40, T=1.0s
target S1: acc=0.6292, itr=144.64 bits/min
target S2: acc=0.7917, itr=208.95 bits/min
target S3: acc=0.9458, itr=283.91 bits/min
target S4: acc=0.8000, itr=212.58 bits/min
target S5: acc=0.7333, itr=184.55 bits/min
target S6: acc=0.5958, itr=132.74 bits/min
target S7: acc=0.6792, itr=163.25 bits/min
target S8: acc=0.3917, itr=68.45 bits/min
target S9: acc=0.6333, itr=146.15 bits/min
target S10: acc=0.6750, itr=161.67 bits/min
target S11: acc=0.1417, itr=11.80 bits/min
target S12: acc=0.7958, itr=210.76 bits/min
target S13: acc=0.4792, itr=94.22 bits/min
target S14: acc=0.7708, itr=200.05 bits/min
target S15: acc=0.4875, itr=96.82 bits/min
target S16: acc=0.4250, itr=77.95 bits/min
target S17: acc=0.4125, itr=74.34 bits/min
target S18: acc=0.7208, itr=179.53 bits/min
target S19: acc=0.1375, itr=11.14 bits/min
target S20: acc=0.5375, itr=112.89 bits/min
target S21: acc=0.4292, itr=79.16 bits/min
target S22: acc=0.8667, itr=

In [3]:
# IISTLF
def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(
            dataset=dataset,
            subjects=[s],
            return_concat=True,
            verbose=False,
        )
        X = np.nan_to_num(
            np.asarray(X, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


def pick_per_stimulus_trials(
    X,
    y,
    n_classes,
    n_calib_per_stimulus=2,
    random_state=42,
):
    """每个刺激类别抽取相同数量的校准 trial。"""
    rng = np.random.RandomState(random_state)
    selected = []

    for cls in range(n_classes):
        idx_cls = np.flatnonzero(y == cls)
        if len(idx_cls) < n_calib_per_stimulus:
            raise ValueError(
                f"目标被试类别 {cls} trial 不足: "
                f"{len(idx_cls)} < {n_calib_per_stimulus}"
            )
        selected.extend(
            rng.choice(
                idx_cls,
                size=n_calib_per_stimulus,
                replace=False,
            )
        )

    selected = np.sort(np.asarray(selected, dtype=int))
    mask = np.ones(len(y), dtype=bool)
    mask[selected] = False

    X_cal = X[selected]
    y_cal = y[selected]
    X_test = X[mask]
    y_test = y[mask]

    if len(X_test) == 0:
        raise ValueError("校准后没有剩余测试 trial。")
    if np.intersect1d(selected, np.flatnonzero(mask)).size:
        raise AssertionError("calibration/test selections overlap")
    if any(np.sum(y_cal == cls) != n_calib_per_stimulus for cls in range(n_classes)):
        raise AssertionError("每个刺激类别的校准 trial 数量不一致")

    return X_cal, y_cal, X_test, y_test


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 主评估：全体 LOSO（不分组）
# =========================
all_subjects = list(range(1, 36))
n_calib_per_stimulus = 1
random_state = 42

n_classes = len(freq_list)
t_sec = 1.0

print("\n========== LOSO on ALL subjects (IISTLF) ==========")
print(f"n_classes={n_classes}, t_sec={t_sec}")

all_data = build_subject_data(Bench_paradigm, Bench_dataset, all_subjects)
all_results = []
accs = []
itrs = []

for target_sid in all_subjects:
    X_t, y_t = all_data[target_sid]

    X_cal, y_cal, X_test, y_test = pick_per_stimulus_trials(
        X_t, y_t,
        n_classes=n_classes,
        n_calib_per_stimulus=n_calib_per_stimulus,
        random_state=random_state,
    )

    source_sids = [s for s in all_subjects if s != target_sid]
    src_accs = []

    for s_sid in source_sids:
        X_s, y_s = all_data[s_sid]

        model = IISTLF(
            srate=sfreq,
            freqs=[float(f) for f in freq_list],
            n_harmonics=3,
            n_subbands=5,          # 与 Table 1 披露的 IISTLF 子带数一致
        )
        model.fit(
            source_Xy=(X_s, y_s.astype(int)),
            target_calib_X=X_cal,
            target_calib_y=y_cal,
        )
        #   y_pred, scores = model.predict(X_test, return_scores=True)
        y_pred = model.predict(X_test)
        src_accs.append(float(np.mean(y_pred == y_test)))

    acc = float(np.mean(src_accs))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=t_sec))

    accs.append(acc)
    itrs.append(itr)

    print(
        f"target S{target_sid}: acc={acc:.4f}, "
        f"itr={itr:.2f} bits/min "
        f"(avg over {len(source_sids)} sources)"
    )

    all_results.append(
        {
            "subject_id": target_sid,
            "subject": f"S{target_sid}",
            "acc": acc,
            "itr": itr,
        }
    )

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


========== LOSO on ALL subjects (IISTLF) ==========
n_classes=40, t_sec=1.0
target S1: acc=0.6596, itr=155.84 bits/min (avg over 34 sources)
target S2: acc=0.7790, itr=203.50 bits/min (avg over 34 sources)
target S3: acc=0.9284, itr=274.29 bits/min (avg over 34 sources)
target S4: acc=0.8888, itr=253.85 bits/min (avg over 34 sources)
target S5: acc=0.8132, itr=218.41 bits/min (avg over 34 sources)
target S6: acc=0.6318, itr=145.58 bits/min (avg over 34 sources)
target S7: acc=0.6284, itr=144.35 bits/min (avg over 34 sources)
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
target S8: acc=0.4196, itr=76.37 bits/min (avg over 34 sources)
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z

In [3]:
# **************************************************
# Benchmark数据集 eTRCA方法 - 全体数据LOSO实验（不分组，含ITR）
# **************************************************
# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ ITR bits/min """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


# ========== eTRCA 固定参数配置 ==========
srate = 250
T = 1
n_harmonics = 3

# 滤波器组配置
wp = [(5, 90)]
ws = [(3, 92)]
filterbank = generate_filterbank(wp, ws, srate=srate, order=15, rp=0.5)

# 获取频率列表
events = Bench_dataset.events.keys()
freq_list = [Bench_dataset.get_freq(event) for event in events]
n_classes = len(freq_list)

# 生成参考信号
Yf = generate_cca_references(freq_list, srate=srate, T=T, n_harmonics=n_harmonics)
print(f"📊 CCA reference shape: {Yf.shape}")
print(f"📊 Number of classes: {n_classes}")
print(f"📊 Decision time: {T}s")

# ========== 一次性获取全体数据 ==========
all_subjects = list(range(1, 36))
X_all, y_all, meta_all = Bench_paradigm.get_data(
    Bench_dataset,
    subjects=all_subjects,          # 全体被试
    return_concat=True,
    n_jobs=None,
    verbose=False
)

print(f"\n📐 Data shape: {X_all.shape}")

all_subjects = np.unique(meta_all['subject'])
N = len(all_subjects)

print(f"👥 Total subjects: {N}")
print(f"📋 Subject IDs: {all_subjects}")

# ========== LOSO ==========
acc_list = []
itr_list = []
subject_results = []

for i, test_subject in enumerate(all_subjects):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    test_mask = (meta_all['subject'] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"📊 Training samples: {len(train_ind)}")
    print(f"📊 Testing samples: {len(test_ind)}")

    estimator = SC_TRCA(
        standard=False,
        ensemble=True,
        n_components=1
    )

    print("🔄 SC_TRCA训练中...")
    estimator.fit(X_train, y_train, Yf)
    print("✅ SC_TRCA训练完成")

    _, p_labels = estimator.predict(X_test)

    n_correct = int(np.sum(p_labels == y_test))
    acc_s = n_correct / len(y_test)
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=T))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    subject_results.append({
        "test_subject": int(test_subject),
        "accuracy": float(acc_s),
        "itr": float(itr_s),
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# ========== 汇总 ==========
acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

df_results = pd.DataFrame(subject_results).sort_values("test_subject")

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)


📊 CCA reference shape: (40, 6, 250)
📊 Number of classes: 40
📊 Decision time: 1s
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yij

In [4]:
# **************************************************
# Benchmark数据集 TtCCA - 全体数据LOSO实验（不分组，含ITR）
# **************************************************

# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ ITR bits/min """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


# ========== 固定参数 ==========
srate = 250
T = 1
n_harmonics = 3

events = Bench_dataset.events.keys()
freq_list = [Bench_dataset.get_freq(event) for event in events]
n_classes = len(freq_list)

# 生成CCA参考信号模板
cca_template = generate_cca_references(freq_list, srate=srate, T=T, n_harmonics=n_harmonics)
print(f"📊 CCA template shape: {cca_template.shape}")
print(f"📊 Number of classes: {n_classes}")
print(f"📊 Decision time: {T}s")


# ========== 一次性获取全体数据（不分组） ==========
all_subjects = list(range(1, 36))
X_all, y_all, meta_all = Bench_paradigm.get_data(
    Bench_dataset,
    subjects=all_subjects,          # 全体被试
    return_concat=True,
    n_jobs=None,
    verbose=False
)

print(f"\n📐 Data shape: {X_all.shape}")

all_subjects = np.unique(meta_all['subject'])
N = len(all_subjects)
print(f"👥 Total subjects: {N}")
print(f"📋 Subject IDs: {all_subjects}")

# ========== LOSO循环 ==========
acc_list = []
itr_list = []
subject_results = []

for i, test_subject in enumerate(all_subjects):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    # LOSO划分
    test_mask = (meta_all['subject'] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"📊 Training samples: {len(train_ind)}")
    print(f"📊 Testing samples: {len(test_ind)}")

    # 初始化TtCCA模型
    estimator = TtCCA(
        n_components=1,
        n_jobs=-1
    )

    # 训练
    print("🔄 TtCCA训练中...")
    estimator.fit(X_train, y_train, cca_template, n_harmonics)
    print("✅ TtCCA训练完成")

    # 预测
    p_labels = estimator.predict(X_test)

    # 指标
    n_correct = int(np.sum(p_labels == y_test))
    acc_s = n_correct / len(y_test)
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=T))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    subject_results.append({
        "test_subject": int(test_subject),
        "accuracy": float(acc_s),
        "itr": float(itr_s),
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# ========== 汇总 ==========
acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

df_results = pd.DataFrame(subject_results).sort_values("test_subject")

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)


📊 CCA template shape: (40, 6, 250)
📊 Number of classes: 40
📊 Decision time: 1s
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yiju

In [ ]:
# TDCA
# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 参数区（可按需调整）
# =========================
n_harmonics = 3
window_length = 1      # 时间窗（秒）
ref_T = 0.9              # 参考信号长度从0.5改为0.4，给padding留余量
padding_len = 1         
n_components = 4

# ITR参数
n_classes = len(freq_list)
t_sec = window_length

print(f"[Benchmark] n_classes={n_classes}, t_sec={t_sec:.2f}s")

# =========================
# 参考信号（TDCA用）
# =========================
Yf = generate_cca_references(
    [float(f) for f in freq_list],
    srate=sfreq,
    T=ref_T,
    n_harmonics=n_harmonics
)
print(f"[Benchmark] CCA references shape: {Yf.shape}")

# =========================
# 一次性加载全体数据（LOSO）
# =========================
X_all, y_all, meta_all = Bench_paradigm.get_data(
    Bench_dataset,
    subjects=all_subjects,
    return_concat=True,
    n_jobs=None,
    verbose=False
)

all_subject_ids = np.unique(meta_all["subject"])
N = len(all_subject_ids)

print("\n" + "#" * 70)
print("# 🔬 TDCA LOSO on ALL subjects")
print("#" * 70)
print(f"Total subjects: {N}")
print(f"Subject IDs: {all_subject_ids}")
print(f"Data shape: {X_all.shape}")

# =========================
# LOSO循环
# =========================
acc_list = []
itr_list = []
results = []

for i, test_subject in enumerate(all_subject_ids):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    test_mask = (meta_all["subject"] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"Training samples: {len(train_ind)}")
    print(f"Testing samples: {len(test_ind)}")
    print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

    estimator = TDCA(
        padding_len=padding_len,
        n_components=n_components
    )

    # -------- 安全检查：避免 again 出现 length 报错 --------
    x_len = X_train.shape[-1]
    yf_len = Yf.shape[-1]   # 参考信号时间点
    if x_len <= (padding_len + yf_len):
        raise ValueError(
            f"Not enough time points for TDCA: "
            f"X_len={x_len}, padding_len={padding_len}, Yf_len={yf_len}, "
            f"need X_len > padding_len + Yf_len.\n"
            f"Try smaller ref_T, smaller padding_len, or longer data window."
        )

    p_labels = estimator.fit(
        X=X_train,
        y=y_train,
        Yf=Yf
    ).predict(X_test)

    acc_s = float(np.mean(p_labels == y_test))
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=t_sec))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    n_correct = int(np.sum(p_labels == y_test))
    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    results.append({
        "test_subject": int(test_subject),
        "accuracy": acc_s,
        "itr": itr_s,
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# =========================
# 汇总打印
# =========================
df_results = pd.DataFrame(results).sort_values("test_subject")

acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)

[Benchmark] n_classes=40, t_sec=1.00s
[Benchmark] CCA references shape: (40, 6, 225)
--------ssssss, /upload/yijun/S1.mat.7z
--------ssssss, /upload/yijun/S2.mat.7z
--------ssssss, /upload/yijun/S3.mat.7z
--------ssssss, /upload/yijun/S4.mat.7z
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /uploa

In [8]:
# EEGconformer
def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(
            dataset=dataset,
            subjects=[s],
            return_concat=True,
            verbose=False,
        )
        X = np.nan_to_num(
            np.asarray(X, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 主评估：全体 LOSO（无目标校准、无目标微调）
# =========================
all_subjects = list(range(1, 36))
random_state = 42
n_classes = len(freq_list)
t_sec = 1.0

print("\n========== LOSO on ALL subjects (EEG-Conformer) ==========")
print(f"n_classes={n_classes}, t_sec={t_sec}")

all_data = build_subject_data(Bench_paradigm, Bench_dataset, all_subjects)
accs = []
itrs = []

for target_sid in all_subjects:
    X_test, y_test = all_data[target_sid]
    train_sids = [s for s in all_subjects if s != target_sid]
    X_train = np.concatenate([all_data[s][0] for s in train_sids], axis=0)
    y_train = np.concatenate([all_data[s][1] for s in train_sids], axis=0)

    model = EEGConformer(
        srate=sfreq,
        freqs=[float(f) for f in freq_list],
        device="cuda" if torch.cuda.is_available() else "cpu",
        emb_size=40,
        depth=3,
        n_heads=4,
        batch_size=256,
        n_epochs=100,
        lr=3e-4,
        beta1=0.5,
        beta2=0.999,
        seed=random_state,
        use_amp=True,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test, batch_size=512)

    acc = float(np.mean(y_pred == y_test))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=t_sec))
    accs.append(acc)
    itrs.append(itr)

    print(f"target S{target_sid}: acc={acc:.4f}, itr={itr:.2f} bits/min")

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


========== LOSO on ALL subjects (EEG-Conformer) ==========
n_classes=40, t_sec=1.0
target S1: acc=0.3958, itr=69.61 bits/min
--------ssssss, /upload/yijun/S5.mat.7z
--------ssssss, /upload/yijun/S6.mat.7z
--------ssssss, /upload/yijun/S7.mat.7z
--------ssssss, /upload/yijun/S8.mat.7z
--------ssssss, /upload/yijun/S9.mat.7z
--------ssssss, /upload/yijun/S10.mat.7z
--------ssssss, /upload/yijun/S11.mat.7z
--------ssssss, /upload/yijun/S12.mat.7z
--------ssssss, /upload/yijun/S13.mat.7z
--------ssssss, /upload/yijun/S14.mat.7z
--------ssssss, /upload/yijun/S15.mat.7z
--------ssssss, /upload/yijun/S16.mat.7z
--------ssssss, /upload/yijun/S17.mat.7z
--------ssssss, /upload/yijun/S18.mat.7z
--------ssssss, /upload/yijun/S19.mat.7z
--------ssssss, /upload/yijun/S20.mat.7z
--------ssssss, /upload/yijun/S21.mat.7z
--------ssssss, /upload/yijun/S22.mat.7z
--------ssssss, /upload/yijun/S23.mat.7z
--------ssssss, /upload/yijun/S24.mat.7z
--------ssssss, /upload/yijun/S25.mat.7z
--------ssssss, /u

In [4]:
#**************************************************
# BETA数据集读取处理
#**************************************************
# BETA数据集，已经通过matlab的eegfit，进行了3-90HZ的带通滤波，故此处不再进行滤波处理
# BETA_wof数据集没有进行滤波处理，链接已经
# 在tsinghua.py把BETA_URL做以下修改即可使用未滤波的版本
# BETA_URL = "https://bci.med.tsinghua.edu.cn/upload/liubingchuan_BETA_wof/"


# 数据存储为一个四维张量 [channel, time point, block, condition]
#                    [   64,       750,      4,      40    ]

BETA_dataset = BETA()
subject_list = list(range(1, 71))  # 被试编号从1到70
for s in subject_list:
    BETA_dataset.data_path(subject=s, path="/home/foam/metabci/mne_data")  # 依次为每个被试设置路径

events = BETA_dataset.events.keys()
freq_list = [str(BETA_dataset.get_freq(event)) for event in events]  # 获得所有刺激的频率
BETA_freq_map = {i: freq for i, freq in enumerate(freq_list)}  # 标签到频率的映射
print(freq_list)  # 输出频率显示

# ---------- Explicit experiment configuration ----------
t_start, t_end = 0.14, 1.14
sfreq = 250
window_length = t_end - t_start
n_samples = int(window_length * sfreq)

print(f"[BETA] Time window: {window_length:.2f}s ({n_samples} samples)")

# add 5-90Hz bandpass filter in raw hook
bandpass_low, bandpass_high = 5, 90
print(f"[BETA] Bandpass filter: {bandpass_low}-{bandpass_high} Hz")

def raw_hook(raw, caches):
    raw.filter(bandpass_low, bandpass_high, l_trans_bandwidth=2, h_trans_bandwidth=5, phase='zero-double')
    caches['raw_stage'] = caches.get('raw_stage', -1) + 1
    return raw, caches


BETA_paradigm = SSVEP(
    channels=['POZ', 'PZ', 'PO3', 'PO5', 'PO4', 'PO6', 'O1', 'OZ', 'O2'],
    intervals=[(t_start, t_end)],  # 提取1秒数据
    events=freq_list,
    srate=sfreq
)
BETA_paradigm.register_raw_hook(raw_hook)
all_subjects = list(range(1, 71))

--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchu

In [7]:
# ============================================================
# 1) Utilities
# ============================================================
def make_harmonic_filterbank(
    srate=250,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
):
    if high_cut >= srate / 2:
        raise ValueError("high_cut must be lower than the Nyquist frequency.")

    harmonic_filterbank = []

    for k in range(1, n_subbands + 1):
        low_cut = low_step * k

        if low_cut >= high_cut:
            raise ValueError("Invalid subband cutoff frequencies.")

        sos = butter(
            N=order,
            Wn=[low_cut, high_cut],
            btype="bandpass",
            fs=srate,
            output="sos",
        )

        harmonic_filterbank.append(sos)
        
    raw_weights = np.asarray(
        [(k ** (-1.25)) + 0.25 for k in range(1, n_subbands + 1)],
        dtype=float,
    )

    harmonic_filterweights = raw_weights / raw_weights.sum()

    return harmonic_filterbank, harmonic_filterweights


harmonic_filterbank, harmonic_filterweights = make_harmonic_filterbank(
    srate=sfreq,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
)

print("Number of harmonic subbands:", len(harmonic_filterbank))
print("Harmonic filter weights:", harmonic_filterweights)

def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.0


# ============================================================
# 3) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 beta 数据；BETA_paradigm 和 BETA_dataset 需已在 Notebook 中定义。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "beta":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []

    for sid in selected_subjects:
        X_s, y_s, _ = BETA_paradigm.get_data(
            BETA_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return subjects_data, subjects_label, subject_ids, freqs_float


# ============================================================
# 4) One target subject
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
):
    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    X_cal, y_cal, X_test, y_test = subject_split_trials(
        X_target,
        y_target,
        n_calib=n_calib,
        rng=rng,
    )

    if len(y_test) == 0:
        return {
            "summary": [],
            "predictions": [],
        }

    source_X = []
    source_y = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue

        sid = subject_ids[src_idx]

        source_X.append(subjects_data[src_idx])
        source_y.append(subjects_label[src_idx])
        source_subjects.append(
            np.full(
                len(subjects_label[src_idx]),
                sid,
                dtype=int,
            )
        )

    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
    )

    calibration_X = X_cal if n_calib > 0 else None

    y_pred = model.predict(
        X_test,
        calib_X=calibration_X,
    )

    # ========================================================
    # 物理频率排序
    # ========================================================
    classes = [
        int(value)
        for value in model.classes_
    ]

    frequencies = [
        float(value)
        for value in model.freqs
    ]

    if len(classes) != len(frequencies):
        raise ValueError(
            "model.classes_ and model.freqs are not aligned."
        )

    frequency_by_class = dict(
        zip(classes, frequencies)
    )

    ordered_classes = sorted(
        classes,
        key=lambda cls: frequency_by_class[cls],
    )

    physical_rank = {
        cls: rank
        for rank, cls in enumerate(ordered_classes)
    }

    # ========================================================
    # Trial-level predictions
    # ========================================================
    prediction_rows = []

    correct = 0
    adjacent_errors = 0
    nonadjacent_errors = 0

    for trial_position, (
        true_class,
        predicted_class,
    ) in enumerate(zip(y_test, y_pred)):

        true_class = int(true_class)
        predicted_class = int(predicted_class)

        is_error = int(true_class != predicted_class)

        error_distance = abs(
            physical_rank[predicted_class]
            - physical_rank[true_class]
        )

        is_adjacent_error = int(
            is_error == 1
            and error_distance == 1
        )

        is_nonadjacent_error = int(
            is_error == 1
            and error_distance > 1
        )

        correct += int(is_error == 0)
        adjacent_errors += is_adjacent_error
        nonadjacent_errors += is_nonadjacent_error

        prediction_rows.append({
            "dataset": run_cfg.dataset_name,
            "window_sec": float(run_cfg.window_sec),
            "variant": variant.name,
            "target_subject_id": target_subject_id,
            "n_calib": int(n_calib),
            "seed": int(seed),
            "trial_position": int(trial_position),
            "true_class": true_class,
            "predicted_class": predicted_class,
            "true_frequency_hz": frequency_by_class[true_class],
            "predicted_frequency_hz": frequency_by_class[predicted_class],
            "true_physical_rank": physical_rank[true_class],
            "predicted_physical_rank": physical_rank[predicted_class],
            "error_distance": int(error_distance),
            "is_error": is_error,
            "is_adjacent_error": is_adjacent_error,
            "is_nonadjacent_error": is_nonadjacent_error,
        })

    n_test = len(y_test)
    total_errors = adjacent_errors + nonadjacent_errors
    acc = correct / n_test

    summary_row = {
        "dataset": run_cfg.dataset_name,
        "window_sec": float(run_cfg.window_sec),
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "acc": float(acc),
        "itr": compute_itr(
            acc,
            n_classes=run_cfg.n_classes,
            t_sec=run_cfg.decision_time,
        ),
        "n_test": int(n_test),
        "adjacent_error_rate": (
            adjacent_errors / n_test
        ),
        "nonadjacent_error_rate": (
            nonadjacent_errors / n_test
        ),
        "adjacent_error_share": (
            adjacent_errors / total_errors
            if total_errors > 0
            else 0.0
        ),
        "stc_split_mode": model.stc_split_mode,
        "enable_stage1": bool(model.enable_stage1),
        "enable_harmonic_branch": bool(
            model.enable_harmonic_branch
        ),
        "harmonic_filterbank_enabled": bool(
            model.harmonic_filterbank is not None
        ),
        "source_score_mode": model.source_score_mode,
        "confidence_lambda": float(
            model.confidence_lambda
        ),
    }

    return {
        "summary": [summary_row],
        "predictions": prediction_rows,
    }


# ============================================================
# 5) Parallel full LOSO
# ============================================================
def run_loso_variants_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):
    subjects_data, subjects_label, subject_ids, freqs_float = (
        load_dataset_for_loso_by_subjects(
            run_cfg.dataset_name,
            subjects,
        )
    )

    n_subj = len(subjects_data)

    summary_rows = []
    prediction_rows = []

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:

            seeds = (
                [0]
                if n_calib == 0
                else run_cfg.repeat_seeds
            )

            for seed in seeds:

                outputs = Parallel(
                    n_jobs=n_jobs,
                    backend=backend,
                    prefer=prefer,
                    verbose=verbose,
                )(
                    delayed(_run_one_target_task)(
                        tgt_idx=tgt_idx,
                        n_subj=n_subj,
                        subject_ids=subject_ids,
                        subjects_data=subjects_data,
                        subjects_label=subjects_label,
                        n_calib=n_calib,
                        seed=seed,
                        variant=variant,
                        base_model_kwargs=base_model_kwargs,
                        freqs_float=freqs_float,
                        run_cfg=run_cfg,
                    )
                    for tgt_idx in range(n_subj)
                )

                for output in outputs:
                    summary_rows.extend(
                        output["summary"]
                    )
                    prediction_rows.extend(
                        output["predictions"]
                    )

    return (
        pd.DataFrame(summary_rows),
        pd.DataFrame(prediction_rows),
    )


# ============================================================
# 6) Ablation variants
# ============================================================
# 当前默认列表对应：
#   - full：正式 time_ordered 方法
#   - stc_random：1.0 s STC random sensitivity
#   - w/o_harmonic：harmonic branch 关闭
#
# 如果要继续运行原来的其他消融，可将下面注释部分加入 variants。
variants = [
    # ========================================================
    # A. Stage 1/JFPM 实验
    # ========================================================
    VariantConfig("stage1_full",{"use_revised_method": True,"enable_stage1": True,"stc_split_mode": "time_ordered",},),
    VariantConfig("w/o_stage1",{"use_revised_method": True,"enable_stage1": False,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_radius0",{"use_revised_method": True,"enable_stage1": True,"neighbor_radius": 0,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_strength0",{"use_revised_method": True,"enable_stage1": True,"neighbor_strength": 0.0,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_strength0.5",{"use_revised_method": True,"enable_stage1": True,"neighbor_strength": 0.5,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_strength2",{"use_revised_method": True,"enable_stage1": True,"neighbor_strength": 2.0,"stc_split_mode": "time_ordered",},),
    VariantConfig("stage1_radius2",{"use_revised_method": True,"enable_stage1": True,"neighbor_radius": 2,"stc_split_mode": "time_ordered",},),

    # ========================================================
    # B. confidence_lambda 敏感性
    # 重点观察 n_calib=2
    # ========================================================
    VariantConfig("lambda_0.0",{"confidence_lambda": 0.0,"stc_split_mode": "time_ordered",},),
    VariantConfig("lambda_0.1",{"confidence_lambda": 0.1,"stc_split_mode": "time_ordered",},),
    VariantConfig("lambda_0.2",{"confidence_lambda": 0.2,"stc_split_mode": "time_ordered",},),
    VariantConfig("lambda_0.5",{"confidence_lambda": 0.5,"stc_split_mode": "time_ordered",},),
    VariantConfig("lambda_1.0",{"confidence_lambda": 1.0,"stc_split_mode": "time_ordered",},),

    # ========================================================
    # C. Harmonic branch 实验
    # ========================================================
    VariantConfig("harmonic_raw",{"enable_harmonic_branch": True,"harmonic_filterbank": None,"harmonic_filterweights": None,"stc_split_mode": "time_ordered",},),
    VariantConfig("harmonic_filterbank",{"enable_harmonic_branch": True,"harmonic_filterbank": harmonic_filterbank,"harmonic_filterweights": harmonic_filterweights,"stc_split_mode": "time_ordered",},),
    VariantConfig("w/o_harmonic",{"enable_harmonic_branch": False,"stc_split_mode": "time_ordered",},),
    # ========================================================
    # D. STC split 敏感性
    # ========================================================
    VariantConfig("stc_time_ordered",{"stc_split_mode": "time_ordered",},),
    VariantConfig("stc_random",{"stc_split_mode": "random",},),
]


# ============================================================
# 7) Run
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.0

base_model_kwargs = dict(
    # 修订版 Stage 1
    use_revised_method=True,
    enable_stage1=True,

    neighbor_radius=1,
    neighbor_decay=0.5,
    neighbor_strength=1.0,

    # 修订版 source score
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,

    # calibration-only Stage 2
    target_alignment_mode="calibration",
    enable_stage2=True,

    # 主方法默认开启 harmonic branch
    enable_harmonic_branch=True,

    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)


run_cfg = RunConfig(
    dataset_name="beta",
    n_calib_list=[0, 1, 2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
)

df_raw, df_predictions = run_loso_variants_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError("df_raw is empty.")

if df_predictions.empty:
    raise RuntimeError("df_predictions is empty.")


# ============================================================
# 8) Summary
# ============================================================

df_summary = (
    df_raw.groupby(
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
        ],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        adjacent_error_rate_mean=(
            "adjacent_error_rate",
            "mean",
        ),
        nonadjacent_error_rate_mean=(
            "nonadjacent_error_rate",
            "mean",
        ),
        n_runs=("acc", "count"),
    )
    .sort_values(
        [
            "variant",
            "n_calib",
        ]
    )
)


# ============================================================
# 9) Paired Wilcoxon + 95% CI
# ============================================================

def paired_comparison(
    df,
    left_variant,
    right_variant,
):
    key_cols = [
        "dataset",
        "window_sec",
        "target_subject_id",
        "n_calib",
        "seed",
    ]

    left = df[
        df["variant"] == left_variant
    ][key_cols + ["acc"]].rename(
        columns={"acc": "acc_left"}
    )

    right = df[
        df["variant"] == right_variant
    ][key_cols + ["acc"]].rename(
        columns={"acc": "acc_right"}
    )

    merged = left.merge(
        right,
        on=key_cols,
        how="inner",
    )

    rows = []

    for (
        dataset,
        window,
        n_calib,
    ), group in merged.groupby(
        [
            "dataset",
            "window_sec",
            "n_calib",
        ]
    ):
        delta = (
            group["acc_left"].to_numpy()
            - group["acc_right"].to_numpy()
        )

        delta = delta[np.isfinite(delta)]

        if len(delta) < 2:
            continue

        try:
            p_value = float(
                wilcoxon(
                    delta,
                    alternative="two-sided",
                ).pvalue
            )
        except ValueError:
            p_value = 1.0

        mean_delta = float(np.mean(delta))
        std_delta = float(np.std(delta, ddof=1))

        critical = float(
            t.ppf(
                0.975,
                df=len(delta) - 1,
            )
        )

        ci_half = (
            critical
            * std_delta
            / np.sqrt(len(delta))
        )

        rows.append({
            "comparison": (
                f"{left_variant} - {right_variant}"
            ),
            "dataset": dataset,
            "window_sec": float(window),
            "n_calib": int(n_calib),
            "n_pairs": int(len(delta)),
            "mean_delta_acc": mean_delta,
            "ci95_low": mean_delta - ci_half,
            "ci95_high": mean_delta + ci_half,
            "p_wilcoxon": p_value,
        })

    return rows


comparison_pairs = [
    ("stage1_full", "w/o_stage1"),
    ("harmonic_filterbank", "harmonic_raw"),
    ("harmonic_raw", "w/o_harmonic"),
    ("stc_random", "stc_time_ordered"),
    ("lambda_0.2", "lambda_0.0"),
]

available_variants = set(
    df_raw["variant"].unique()
)

paired_rows = []

for left_variant, right_variant in comparison_pairs:
    if {
        left_variant,
        right_variant,
    }.issubset(available_variants):

        paired_rows.extend(
            paired_comparison(
                df_raw,
                left_variant,
                right_variant,
            )
        )

df_paired = pd.DataFrame(paired_rows)


def holm_adjust(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)

    running = 0.0
    m = len(p_values)

    for rank, index in enumerate(order):
        value = min(
            1.0,
            p_values[index] * (m - rank),
        )
        running = max(running, value)
        adjusted[index] = running

    return adjusted


if not df_paired.empty:
    df_paired["p_holm"] = holm_adjust(
        df_paired["p_wilcoxon"].to_numpy()
    )


# ============================================================
# 10) Adjacent-frequency error rate
# ============================================================

def adjacent_summary(group):
    n_trials = len(group)
    n_errors = int(group["is_error"].sum())
    n_adjacent = int(
        group["is_adjacent_error"].sum()
    )
    n_nonadjacent = int(
        group["is_nonadjacent_error"].sum()
    )

    return pd.Series({
        "n_trials": n_trials,
        "accuracy": 1.0 - n_errors / n_trials,
        "adjacent_error_rate": (
            n_adjacent / n_trials
        ),
        "nonadjacent_error_rate": (
            n_nonadjacent / n_trials
        ),
        "adjacent_error_share": (
            n_adjacent / n_errors
            if n_errors > 0
            else 0.0
        ),
    })


df_adjacent = (
    df_predictions
    .groupby(
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
        ]
    )
    .apply(adjacent_summary)
    .reset_index()
)


# ============================================================
# 11) Complete confusion matrices
# ============================================================

confusion_rows = []

for group_keys, group in df_predictions.groupby(
    [
        "dataset",
        "window_sec",
        "variant",
        "n_calib",
    ]
):
    (
        dataset,
        window,
        variant,
        n_calib,
    ) = group_keys

    classes = sorted(
        set(group["true_class"])
        | set(group["predicted_class"])
    )

    counts = (
        group
        .groupby(
            [
                "true_class",
                "predicted_class",
            ]
        )
        .size()
        .to_dict()
    )

    for true_class in classes:
        for predicted_class in classes:
            confusion_rows.append({
                "dataset": dataset,
                "window_sec": float(window),
                "variant": variant,
                "n_calib": int(n_calib),
                "true_class": int(true_class),
                "predicted_class": int(
                    predicted_class
                ),
                "count": int(
                    counts.get(
                        (
                            true_class,
                            predicted_class,
                        ),
                        0,
                    )
                ),
            })


df_confusion = pd.DataFrame(confusion_rows)


# Wide-format confusion matrix for convenient plotting
df_confusion_wide = (
    df_confusion
    .pivot_table(
        index=[
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
            "true_class",
        ],
        columns="predicted_class",
        values="count",
        fill_value=0,
    )
    .reset_index()
)


# ============================================================
# 12) Save all detailed results
# ============================================================

df_raw.to_csv(
    f"subject_accuracy_{window_sec:.1f}s.csv",
    index=False,
)

df_summary.to_csv(
    f"summary_{window_sec:.1f}s.csv",
    index=False,
)

df_predictions.to_csv(
    f"trial_predictions_{window_sec:.1f}s.csv",
    index=False,
)

df_paired.to_csv(
    f"paired_wilcoxon_ci_holm_{window_sec:.1f}s.csv",
    index=False,
)

df_adjacent.to_csv(
    f"adjacent_frequency_errors_{window_sec:.1f}s.csv",
    index=False,
)

df_confusion.to_csv(
    f"confusion_matrices_long_{window_sec:.1f}s.csv",
    index=False,
)

df_confusion_wide.to_csv(
    f"confusion_matrices_wide_{window_sec:.1f}s.csv",
    index=False,
)


# ============================================================
# 13) Concise printing
# ============================================================

print("\n========== Key Accuracy Results ==========")

key_variants = [
    "stage1_full",
    "w/o_stage1",
    "harmonic_raw",
    "harmonic_filterbank",
    "w/o_harmonic",
    "stc_time_ordered",
    "stc_random",
    "lambda_0.0",
    "lambda_0.2",
    "lambda_1.0",
]

print(
    df_summary[
        df_summary["variant"].isin(key_variants)
    ][
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
            "acc_mean",
            "itr_mean",
        ]
    ].to_string(index=False)
)

print("\n========== Paired Statistics: n_calib=2 ==========")

if not df_paired.empty:
    print(
        df_paired[
            df_paired["n_calib"] == 2
        ][
            [
                "comparison",
                "dataset",
                "window_sec",
                "mean_delta_acc",
                "ci95_low",
                "ci95_high",
                "p_wilcoxon",
                "p_holm",
            ]
        ].to_string(index=False)
    )

print("\n========== Adjacent-Frequency Errors: n_calib=2 ==========")

print(
    df_adjacent[
        df_adjacent["n_calib"] == 2
    ].to_string(index=False)
)
print("\nDetailed CSV files saved.")


Number of harmonic subbands: 5
Harmonic filter weights: [0.38648818 0.20729625 0.15560897 0.13195532 0.11865127]
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_w

In [6]:
# ============================================================
# 1) Utilities
# ============================================================
def make_harmonic_filterbank(
    srate=250,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
):
    if high_cut >= srate / 2:
        raise ValueError("high_cut must be lower than the Nyquist frequency.")

    harmonic_filterbank = []

    for k in range(1, n_subbands + 1):
        low_cut = low_step * k

        if low_cut >= high_cut:
            raise ValueError("Invalid subband cutoff frequencies.")

        sos = butter(
            N=order,
            Wn=[low_cut, high_cut],
            btype="bandpass",
            fs=srate,
            output="sos",
        )

        harmonic_filterbank.append(sos)
        
    raw_weights = np.asarray(
        [(k ** (-1.25)) + 0.25 for k in range(1, n_subbands + 1)],
        dtype=float,
    )

    harmonic_filterweights = raw_weights / raw_weights.sum()

    return harmonic_filterbank, harmonic_filterweights


harmonic_filterbank, harmonic_filterweights = make_harmonic_filterbank(
    srate=sfreq,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
)

print("Number of harmonic subbands:", len(harmonic_filterbank))
print("Harmonic filter weights:", harmonic_filterweights)

def subject_split_indices(y_subj, n_calib, rng):
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    return (
        np.asarray(calib_idx, dtype=int),
        np.asarray(test_idx, dtype=int),
    )


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.0


# ============================================================
# 3) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 BETA 数据及与 trial 顺序对齐的真实 session/run block 标签。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "beta":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []
    block_ids_by_subject = {}

    for sid in selected_subjects:
        X_s, y_s, meta_s = BETA_paradigm.get_data(
            BETA_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        if not isinstance(meta_s, pd.DataFrame):
            raise TypeError(
                f"Subject {sid}: expected pandas DataFrame metadata, "
                f"got {type(meta_s).__name__}."
            )
        if len(meta_s) != len(y_s):
            raise ValueError(
                f"Subject {sid}: metadata/trial length mismatch: "
                f"{len(meta_s)} != {len(y_s)}."
            )
        if not {"session", "run"}.issubset(meta_s.columns):
            raise ValueError(
                f"Subject {sid}: metadata must contain session and run. "
                f"Available columns: {meta_s.columns.tolist()}"
            )
        if meta_s[["session", "run"]].isna().any().any():
            raise ValueError(
                f"Subject {sid}: session/run metadata contains missing values."
            )

        # meta_s is returned in the same epoch order as X_s/y_s.  The pair
        # (session, run) therefore gives one real acquisition block label per
        # trial, without inferring any block from trial indices.
        block_ids_by_subject[sid] = np.asarray(
            [
                f"subject={sid}|session={session}|run={run}"
                for session, run in zip(meta_s["session"], meta_s["run"])
            ],
            dtype=object,
        )

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return (
        subjects_data,
        subjects_label,
        subject_ids,
        freqs_float,
        block_ids_by_subject,
    )


# ============================================================
# 4) One target subject
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
    block_ids_by_subject=None,
):

    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    calib_idx, test_idx = subject_split_indices(
        y_target,
        n_calib=n_calib,
        rng=rng,
    )
    target_subject_id = subject_ids[tgt_idx]

    target_block_ids = None
    if block_ids_by_subject is not None:
        if target_subject_id not in block_ids_by_subject:
            raise KeyError(
                f"Missing block IDs for target subject {target_subject_id}"
            )
    
        target_block_ids = np.asarray(
            block_ids_by_subject[target_subject_id],
            dtype=object,
        )
    
        if len(target_block_ids) != len(y_target):
            raise ValueError(
                f"Target subject {target_subject_id}: "
                "block_ids length does not match trial count."
            )
    
    target_calib_block_ids = (
        target_block_ids[calib_idx]
        if target_block_ids is not None and n_calib > 0
        else None
    )
    
    X_cal = X_target[calib_idx]
    y_cal = y_target[calib_idx]
    X_test = X_target[test_idx]
    y_test = y_target[test_idx]


    if len(y_test) == 0:
        return {
            "summary": [],
            "predictions": [],
        }

    source_X = []
    source_y = []
    source_block_ids_list = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue
    
        sid = subject_ids[src_idx]
        X_source_subject = subjects_data[src_idx]
        y_source_subject = subjects_label[src_idx]
    
        source_X.append(X_source_subject)
        source_y.append(y_source_subject)
        source_subjects.append(
            np.full(len(y_source_subject), sid, dtype=int)
        )
    
        if block_ids_by_subject is not None:
            if sid not in block_ids_by_subject:
                raise KeyError(f"Missing block IDs for source subject {sid}")
    
            sid_block_ids = np.asarray(
                block_ids_by_subject[sid],
                dtype=object,
            )
    
            if len(sid_block_ids) != len(y_source_subject):
                raise ValueError(
                    f"Source subject {sid}: "
                    "block_ids length does not match trial count."
                )
    
            source_block_ids_list.append(sid_block_ids)


    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)
    source_block_ids = (
        np.concatenate(source_block_ids_list, axis=0)
        if source_block_ids_list
        else None
    )

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    if (model_kwargs.get("stc_split_mode") == "block_stratified"
        and source_block_ids is None
    ):
        raise ValueError(
            "block_stratified requires real source block/run labels."
        )
    
    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
        source_block_ids=source_block_ids,
        target_calib_block_ids=target_calib_block_ids,
    )


    calibration_X = X_cal if n_calib > 0 else None

    y_pred = model.predict(
        X_test,
        calib_X=calibration_X,
    )

    # ========================================================
    # 物理频率排序
    # ========================================================
    classes = [
        int(value)
        for value in model.classes_
    ]

    frequencies = [
        float(value)
        for value in model.freqs
    ]

    if len(classes) != len(frequencies):
        raise ValueError(
            "model.classes_ and model.freqs are not aligned."
        )

    frequency_by_class = dict(
        zip(classes, frequencies)
    )

    ordered_classes = sorted(
        classes,
        key=lambda cls: frequency_by_class[cls],
    )

    physical_rank = {
        cls: rank
        for rank, cls in enumerate(ordered_classes)
    }

    # ========================================================
    # Trial-level predictions
    # ========================================================
    prediction_rows = []

    correct = 0
    adjacent_errors = 0
    nonadjacent_errors = 0

    for trial_position, (
        true_class,
        predicted_class,
    ) in enumerate(zip(y_test, y_pred)):

        true_class = int(true_class)
        predicted_class = int(predicted_class)

        is_error = int(true_class != predicted_class)

        error_distance = abs(
            physical_rank[predicted_class]
            - physical_rank[true_class]
        )

        is_adjacent_error = int(
            is_error == 1
            and error_distance == 1
        )

        is_nonadjacent_error = int(
            is_error == 1
            and error_distance > 1
        )

        correct += int(is_error == 0)
        adjacent_errors += is_adjacent_error
        nonadjacent_errors += is_nonadjacent_error

        prediction_rows.append({
            "dataset": run_cfg.dataset_name,
            "window_sec": float(run_cfg.window_sec),
            "variant": variant.name,
            "target_subject_id": target_subject_id,
            "n_calib": int(n_calib),
            "seed": int(seed),
            "trial_position": int(trial_position),
            "true_class": true_class,
            "predicted_class": predicted_class,
            "true_frequency_hz": frequency_by_class[true_class],
            "predicted_frequency_hz": frequency_by_class[predicted_class],
            "true_physical_rank": physical_rank[true_class],
            "predicted_physical_rank": physical_rank[predicted_class],
            "error_distance": int(error_distance),
            "is_error": is_error,
            "is_adjacent_error": is_adjacent_error,
            "is_nonadjacent_error": is_nonadjacent_error,
        })

    n_test = len(y_test)
    total_errors = adjacent_errors + nonadjacent_errors
    acc = correct / n_test

    summary_row = {
        "dataset": run_cfg.dataset_name,
        "window_sec": float(run_cfg.window_sec),
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "acc": float(acc),
        "itr": compute_itr(
            acc,
            n_classes=run_cfg.n_classes,
            t_sec=run_cfg.decision_time,
        ),
        "n_test": int(n_test),
        "adjacent_error_rate": (
            adjacent_errors / n_test
        ),
        "nonadjacent_error_rate": (
            nonadjacent_errors / n_test
        ),
        "adjacent_error_share": (
            adjacent_errors / total_errors
            if total_errors > 0
            else 0.0
        ),
        "stc_split_mode": model.stc_split_mode,
        "enable_stage1": bool(model.enable_stage1),
        "enable_harmonic_branch": bool(
            model.enable_harmonic_branch
        ),
        "harmonic_filterbank_enabled": bool(
            model.harmonic_filterbank is not None
        ),
        "source_score_mode": model.source_score_mode,
        "confidence_lambda": float(
            model.confidence_lambda
        ),
    }

    return {
        "summary": [summary_row],
        "predictions": prediction_rows,
    }


# ============================================================
# 5) Parallel full LOSO
# ============================================================
def run_loso_variants_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    block_ids_by_subject=None,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):

    (
        subjects_data,
        subjects_label,
        subject_ids,
        freqs_float,
        loaded_block_ids_by_subject,
    ) = (
        load_dataset_for_loso_by_subjects(
            run_cfg.dataset_name,
            subjects,
        )
    )

    # 默认使用 loader 从真实 metadata 提取的 (session, run) 标签；
    # 仅在调用者显式传入时才使用外部映射。
    if block_ids_by_subject is None:
        block_ids_by_subject = loaded_block_ids_by_subject

    block_count_by_subject = {
        sid: int(len(np.unique(block_ids_by_subject[sid])))
        for sid in subject_ids
    }
    print(
        "[block_stratified] real session/run blocks per subject: "
        f"min={min(block_count_by_subject.values())}, "
        f"max={max(block_count_by_subject.values())}"
    )

    n_subj = len(subjects_data)

    summary_rows = []
    prediction_rows = []

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:

            seeds = (
                [0]
                if n_calib == 0
                else run_cfg.repeat_seeds
            )

            for seed in seeds:

                outputs = Parallel(
                    n_jobs=n_jobs,
                    backend=backend,
                    prefer=prefer,
                    verbose=verbose,
                )(
                    delayed(_run_one_target_task)(
                         tgt_idx=tgt_idx,
                        n_subj=n_subj,
                        subject_ids=subject_ids,
                        subjects_data=subjects_data,
                        subjects_label=subjects_label,
                        n_calib=n_calib,
                        seed=seed,
                        variant=variant,
                        base_model_kwargs=base_model_kwargs,
                        freqs_float=freqs_float,
                        run_cfg=run_cfg,
                        block_ids_by_subject=block_ids_by_subject,
                    )
                    for tgt_idx in range(n_subj)
                )

                for output in outputs:
                    summary_rows.extend(
                        output["summary"]
                    )
                    prediction_rows.extend(
                        output["predictions"]
                    )

    return (
        pd.DataFrame(summary_rows),
        pd.DataFrame(prediction_rows),
    )


# ============================================================
# 6) STC split sensitivity only
# ============================================================
# 除 STC split 外，两个条件使用完全相同的修订版 SQ-HAF 设置。
# 不运行 Stage 1、lambda 或 harmonic branch 消融。
variants = [
    VariantConfig(
        "stc_time_ordered",
        {
            "use_revised_method": True,
            "enable_stage1": True,
            "stc_split_mode": "time_ordered",
        },
    ),
    VariantConfig(
        "stc_block_stratified",
        {
            "use_revised_method": True,
            "enable_stage1": True,
            "stc_split_mode": "block_stratified",
        },
    ),
]


# ============================================================
# 7) Run
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.0

base_model_kwargs = dict(
    # 修订版 Stage 1
    use_revised_method=True,
    enable_stage1=True,

    neighbor_radius=1,
    neighbor_decay=0.5,
    neighbor_strength=1.0,

    # 修订版 source score
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,

    # calibration-only Stage 2
    target_alignment_mode="calibration",
    enable_stage2=True,

    # 主协议；block-stratified 仅由对应 sensitivity variant 覆盖。
    stc_split_mode="time_ordered",

    # 主方法默认开启 harmonic branch
    enable_harmonic_branch=True,

    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)


run_cfg = RunConfig(
    dataset_name="beta",
    n_calib_list=[0, 1, 2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
)

df_raw, df_predictions = run_loso_variants_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError("df_raw is empty.")

if df_predictions.empty:
    raise RuntimeError("df_predictions is empty.")

expected_split_modes = {"time_ordered", "block_stratified"}
observed_split_modes = set(
    df_raw["stc_split_mode"].dropna().astype(str).unique()
)
if observed_split_modes != expected_split_modes:
    raise AssertionError(
        "STC sensitivity run must contain time_ordered and block_stratified only; "
        f"observed={observed_split_modes}"
    )


# ============================================================
# 8) Summary
# ============================================================

df_summary = (
    df_raw.groupby(
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
        ],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        adjacent_error_rate_mean=(
            "adjacent_error_rate",
            "mean",
        ),
        nonadjacent_error_rate_mean=(
            "nonadjacent_error_rate",
            "mean",
        ),
        n_runs=("acc", "count"),
    )
    .sort_values(
        [
            "variant",
            "n_calib",
        ]
    )
)


# ============================================================
# 9) Paired Wilcoxon + 95% CI
# ============================================================

def paired_comparison(df, left_variant, right_variant):
    """Paired block-minus-time statistics for ACC and ITR.

    n_calib=1 is retained in the saved table as a descriptive check only:
    the STC confidence gate is zero in that setting, so it is excluded from
    the planned inferential family and Holm correction.
    """
    key_cols = [
        "dataset",
        "window_sec",
        "target_subject_id",
        "n_calib",
        "seed",
    ]

    left = df[df["variant"] == left_variant][
        key_cols + ["acc", "itr"]
    ].rename(
        columns={
            "acc": "acc_left",
            "itr": "itr_left",
        }
    )

    right = df[df["variant"] == right_variant][
        key_cols + ["acc", "itr"]
    ].rename(
        columns={
            "acc": "acc_right",
            "itr": "itr_right",
        }
    )

    merged = left.merge(
        right,
        on=key_cols,
        how="inner",
    )

    rows = []

    for (
        dataset,
        window,
        n_calib,
    ), group in merged.groupby(
        [
            "dataset",
            "window_sec",
            "n_calib",
        ]
    ):
        for metric in ("acc", "itr"):
            delta = (
                group[f"{metric}_left"].to_numpy(dtype=float)
                - group[f"{metric}_right"].to_numpy(dtype=float)
            )
            delta = delta[np.isfinite(delta)]

            if len(delta) < 2:
                continue

            mean_delta = float(np.mean(delta))
            std_delta = float(np.std(delta, ddof=1))
            critical = float(t.ppf(0.975, df=len(delta) - 1))
            ci_half = critical * std_delta / np.sqrt(len(delta))

            inferential = int(n_calib) in {0, 2}
            if not inferential or np.allclose(delta, 0.0):
                p_value = 1.0 if inferential else np.nan
            else:
                try:
                    p_value = float(
                        wilcoxon(delta, alternative="two-sided").pvalue
                    )
                except ValueError:
                    p_value = 1.0

            rows.append({
                "comparison": f"{left_variant} - {right_variant}",
                "metric": metric,
                "dataset": dataset,
                "window_sec": float(window),
                "n_calib": int(n_calib),
                "n_pairs": int(len(delta)),
                "mean_delta": mean_delta,
                "ci95_low": mean_delta - ci_half,
                "ci95_high": mean_delta + ci_half,
                "descriptive_only": not inferential,
                "p_wilcoxon": p_value,
            })

    return rows


comparison_pairs = [
    ("stc_block_stratified", "stc_time_ordered"),
]

available_variants = set(
    df_raw["variant"].unique()
)

paired_rows = []

for left_variant, right_variant in comparison_pairs:
    if {
        left_variant,
        right_variant,
    }.issubset(available_variants):

        paired_rows.extend(
            paired_comparison(
                df_raw,
                left_variant,
                right_variant,
            )
        )

df_paired = pd.DataFrame(paired_rows)


def holm_adjust(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    order = np.argsort(p_values)
    adjusted = np.empty_like(p_values)

    running = 0.0
    m = len(p_values)

    for rank, index in enumerate(order):
        value = min(
            1.0,
            p_values[index] * (m - rank),
        )
        running = max(running, value)
        adjusted[index] = running

    return adjusted


if not df_paired.empty:
    df_paired["p_holm"] = np.nan
    inferential_rows = df_paired["p_wilcoxon"].notna()
    df_paired.loc[inferential_rows, "p_holm"] = holm_adjust(
        df_paired.loc[inferential_rows, "p_wilcoxon"].to_numpy()
    )


# ============================================================
# 10) Adjacent-frequency error rate
# ============================================================

def adjacent_summary(group):
    n_trials = len(group)
    n_errors = int(group["is_error"].sum())
    n_adjacent = int(
        group["is_adjacent_error"].sum()
    )
    n_nonadjacent = int(
        group["is_nonadjacent_error"].sum()
    )

    return pd.Series({
        "n_trials": n_trials,
        "accuracy": 1.0 - n_errors / n_trials,
        "adjacent_error_rate": (
            n_adjacent / n_trials
        ),
        "nonadjacent_error_rate": (
            n_nonadjacent / n_trials
        ),
        "adjacent_error_share": (
            n_adjacent / n_errors
            if n_errors > 0
            else 0.0
        ),
    })


df_adjacent = (
    df_predictions
    .groupby(
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
        ]
    )
    .apply(adjacent_summary)
    .reset_index()
)


# ============================================================
# 11) Complete confusion matrices
# ============================================================

confusion_rows = []

for group_keys, group in df_predictions.groupby(
    [
        "dataset",
        "window_sec",
        "variant",
        "n_calib",
    ]
):
    (
        dataset,
        window,
        variant,
        n_calib,
    ) = group_keys

    classes = sorted(
        set(group["true_class"])
        | set(group["predicted_class"])
    )

    counts = (
        group
        .groupby(
            [
                "true_class",
                "predicted_class",
            ]
        )
        .size()
        .to_dict()
    )

    for true_class in classes:
        for predicted_class in classes:
            confusion_rows.append({
                "dataset": dataset,
                "window_sec": float(window),
                "variant": variant,
                "n_calib": int(n_calib),
                "true_class": int(true_class),
                "predicted_class": int(
                    predicted_class
                ),
                "count": int(
                    counts.get(
                        (
                            true_class,
                            predicted_class,
                        ),
                        0,
                    )
                ),
            })


df_confusion = pd.DataFrame(confusion_rows)


# Wide-format confusion matrix for convenient plotting
df_confusion_wide = (
    df_confusion
    .pivot_table(
        index=[
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
            "true_class",
        ],
        columns="predicted_class",
        values="count",
        fill_value=0,
    )
    .reset_index()
)


# ============================================================
# 12) Save all detailed results
# ============================================================

df_raw.to_csv(
    f"stc_sensitivity_subject_results_{window_sec:.1f}s.csv",
    index=False,
)

df_summary.to_csv(
    f"stc_sensitivity_summary_{window_sec:.1f}s.csv",
    index=False,
)

df_predictions.to_csv(
    f"stc_sensitivity_trial_predictions_{window_sec:.1f}s.csv",
    index=False,
)

df_paired.to_csv(
    f"stc_sensitivity_paired_wilcoxon_ci_holm_{window_sec:.1f}s.csv",
    index=False,
)

df_adjacent.to_csv(
    f"stc_sensitivity_adjacent_frequency_errors_{window_sec:.1f}s.csv",
    index=False,
)

df_confusion.to_csv(
    f"stc_sensitivity_confusion_matrices_long_{window_sec:.1f}s.csv",
    index=False,
)

df_confusion_wide.to_csv(
    f"stc_sensitivity_confusion_matrices_wide_{window_sec:.1f}s.csv",
    index=False,
)


# ============================================================
# 13) Concise printing
# ============================================================

print("\n========== STC Split Sensitivity ==========")

key_variants = [
    "stc_time_ordered",
    "stc_block_stratified",
]

print(
    df_summary[
        df_summary["variant"].isin(key_variants)
    ][
        [
            "dataset",
            "window_sec",
            "variant",
            "n_calib",
            "acc_mean",
            "itr_mean",
        ]
    ].to_string(index=False)
)

print("\n========== Paired Block-Stratified − Time-Ordered ==========")

if not df_paired.empty:
    print(
        df_paired[
            df_paired["n_calib"].isin([0, 2])
        ][
            [
                "comparison",
                "metric",
                "dataset",
                "window_sec",
                "n_calib",
                "n_pairs",
                "mean_delta",
                "ci95_low",
                "ci95_high",
                "p_wilcoxon",
                "p_holm",
            ]
        ].to_string(index=False)
    )

print("\n========== n_calib=1 Descriptive Check ==========")

if not df_paired.empty:
    print(
        df_paired[
            df_paired["n_calib"] == 1
        ][
            [
                "comparison",
                "metric",
                "dataset",
                "window_sec",
                "n_pairs",
                "mean_delta",
                "ci95_low",
                "ci95_high",
                "descriptive_only",
            ]
        ].to_string(index=False)
    )

print("\nSTC sensitivity CSV files saved.")

Number of harmonic subbands: 5
Harmonic filter weights: [0.38648818 0.20729625 0.15560897 0.13195532 0.11865127]
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_w

In [5]:

# ============================================================
# 1) Utilities
# ============================================================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Five-subband harmonic filter bank
# ============================================================
def make_harmonic_filterbank(
    srate=250,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
):
    """
    Build the five-subband filter bank used only by the
    harmonic-reference branch.

    Subbands:
        1:  6-90 Hz
        2: 12-90 Hz
        3: 18-90 Hz
        4: 24-90 Hz
        5: 30-90 Hz

    Normalized weights:
        w_k proportional to k^(-1.25) + 0.25
    """
    if high_cut >= srate / 2:
        raise ValueError("high_cut must be lower than the Nyquist frequency.")

    harmonic_filterbank = []

    for k in range(1, n_subbands + 1):
        low_cut = low_step * k

        if low_cut >= high_cut:
            raise ValueError("Invalid subband cutoff frequencies.")

        sos = butter(
            N=order,
            Wn=[low_cut, high_cut],
            btype="bandpass",
            fs=srate,
            output="sos",
        )
        harmonic_filterbank.append(sos)

    raw_weights = np.asarray(
        [(k ** (-1.25)) + 0.25 for k in range(1, n_subbands + 1)],
        dtype=float,
    )
    harmonic_filterweights = raw_weights / raw_weights.sum()

    return harmonic_filterbank, harmonic_filterweights


# ============================================================
# 3) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.0


# ============================================================
# 4) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 beta 数据；BETA_paradigm 和 BETA_dataset 需已在 Notebook 中定义。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "beta":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []

    for sid in selected_subjects:
        X_s, y_s, _ = BETA_paradigm.get_data(
            BETA_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return subjects_data, subjects_label, subject_ids, freqs_float


# ============================================================
# 5) One target subject
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
):
    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    X_cal, y_cal, X_test, y_test = subject_split_trials(
        X_target,
        y_target,
        n_calib=n_calib,
        rng=rng,
    )

    if len(y_test) == 0:
        return []

    source_X = []
    source_y = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue

        sid = subject_ids[src_idx]
        X_source_subject = subjects_data[src_idx]
        y_source_subject = subjects_label[src_idx]

        source_X.append(X_source_subject)
        source_y.append(y_source_subject)
        source_subjects.append(
            np.full(len(y_source_subject), sid, dtype=int)
        )

    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)

    # 使 target split、STC random split 和 source random selection 使用相同 seed
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
    )

    calibration_X = X_cal if n_calib > 0 else None
    y_pred = model.predict(X_test, calib_X=calibration_X)

    acc = float(np.mean(y_pred == y_test))
    itr = compute_itr(
        acc,
        n_classes=run_cfg.n_classes,
        t_sec=run_cfg.decision_time,
    )

    return [{
        "dataset": run_cfg.dataset_name,
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "acc": acc,
        "itr": itr,
        "n_test": int(len(y_test)),
        "stc_split_mode": model.stc_split_mode,
        "enable_harmonic_branch": bool(model.enable_harmonic_branch),
        "harmonic_n_subbands": (
            0
            if not model.enable_harmonic_branch
            else (
                len(model.harmonic_filterbank)
                if model.harmonic_filterbank is not None
                else 1
            )
        ),
        "enable_stage1": bool(model.enable_stage1),
        "enable_stage2": bool(model.enable_stage2),
        "enable_branch_r2": bool(model.enable_branch_r2),
        "enable_branch_r3": bool(model.enable_branch_r3),
    }]


# ============================================================
# 6) Parallel full LOSO
# ============================================================
def run_loso_variants_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):
    subjects_data, subjects_label, subject_ids, freqs_float = (
        load_dataset_for_loso_by_subjects(
            run_cfg.dataset_name,
            subjects,
        )
    )

    n_subj = len(subjects_data)
    rows = []

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:
            seeds = [0] if n_calib == 0 else run_cfg.repeat_seeds

            for seed in seeds:
                out = Parallel(
                    n_jobs=n_jobs,
                    backend=backend,
                    prefer=prefer,
                    verbose=verbose,
                )(
                    delayed(_run_one_target_task)(
                        tgt_idx=tgt_idx,
                        n_subj=n_subj,
                        subject_ids=subject_ids,
                        subjects_data=subjects_data,
                        subjects_label=subjects_label,
                        n_calib=n_calib,
                        seed=seed,
                        variant=variant,
                        base_model_kwargs=base_model_kwargs,
                        freqs_float=freqs_float,
                        run_cfg=run_cfg,
                    )
                    for tgt_idx in range(n_subj)
                )

                for result in out:
                    if result:
                        rows.extend(result)

    return pd.DataFrame(rows)


# ============================================================
# 7) Ablation variants
# ============================================================
# Every variant below inherits the five-subband harmonic-reference
# baseline from base_model_kwargs. Each ablation changes only the
# explicitly listed parameter(s).
variants = [
    # =====================================================
    # Main method: five-subband harmonic-reference baseline
    # =====================================================
    VariantConfig("full",{
            "source_score_mode": "similarity_confidence",
            "stc_split_mode": "time_ordered",
            "enable_harmonic_branch": True,},),
    # =====================================================
    # Source-selection ablation
    # =====================================================

    # 随机选择源被试
    VariantConfig("random_source_selection",{"source_score_mode": "random","source_weight_mode": "uniform",},),
    # 只按目标-源模板相似度评分，不使用 confidence penalty
    VariantConfig("similarity_only_source_scoring",{"source_score_mode": "similarity",},),
    # 只按源被试稳定性评分
    VariantConfig("stability_only_source_scoring",{"source_score_mode": "stability",},),
    # 保留完整 source ranking，但所有选中源使用相同权重
    VariantConfig("uniform_source_weighting",{"source_weight_mode": "uniform",},),
    #  similarity × STC 方法
    VariantConfig("legacy_similarity_stc",{"source_score_mode": "legacy_multiplicative",},),
    # =====================================================
    # Alignment ablation
    # =====================================================
    # 当前实现中 Stage 2 对应 global covariance alignment
    VariantConfig("w/o_global_alignment",{"enable_stage2": False,},),
    # 当前实现中 Stage 1 对应 hierarchical/local alignment
    VariantConfig("w/o_hierarchical_local_alignment",{"enable_stage1": False,},),
    # =====================================================
    # Branch ablation
    # =====================================================
    # harmonic-reference branch，即 r1
    VariantConfig("w/o_harmonic_reference_branch",{"enable_harmonic_branch": False,},),
    # generalized source-template branch，即 r2
    VariantConfig("w/o_source_template_branch",{"enable_branch_r2": False,},),
    # source-specific template branch，即 r3
    VariantConfig("w/o_source_specific_template_branch",{"enable_branch_r3": False,},),
    # =====================================================
    # Fusion ablation
    # =====================================================
    VariantConfig("absolute_value_fusion",{"fusion_mode": "abs_sum",},),
    # =====================================================
    # Reviewer 4: STC split sensitivity
    # =====================================================
    VariantConfig("stc_random",{"stc_split_mode": "random",},),
]


# ============================================================
# 8) Run
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.0

# ------------------------------------------------------------
# Build the five-subband harmonic-reference baseline.
# IMPORTANT:
#   Use harmonic_filterbank / harmonic_filterweights here.
#   Do NOT assign these filters to the model's main filterbank,
#   because the goal is to change only the harmonic-reference branch.
# ------------------------------------------------------------
harmonic_filterbank, harmonic_filterweights = make_harmonic_filterbank(
    srate=sfreq,
    n_subbands=5,
    order=3,
    high_cut=90.0,
    low_step=6.0,
)

print("Number of harmonic subbands:", len(harmonic_filterbank))
print("Harmonic filter weights:", harmonic_filterweights)

base_model_kwargs = dict(
    # Revised SQ-HAF
    use_revised_method=True,

    # Source selection
    n_sources=10,
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,

    # Stage 1
    neighbor_radius=1,
    neighbor_decay=0.5,
    neighbor_strength=1.0,
    enable_stage1=True,

    # Stage 2
    target_alignment_mode="calibration",
    enable_stage2=True,

    # STC
    stc_split_mode="time_ordered",

    # Three scoring branches
    enable_harmonic_branch=True,
    enable_branch_r2=True,
    enable_branch_r3=True,

    # --------------------------------------------------------
    # Five-subband harmonic-reference branch baseline
    # --------------------------------------------------------
    harmonic_filterbank=harmonic_filterbank,
    harmonic_filterweights=harmonic_filterweights,

    # Sinusoidal references
    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)

run_cfg = RunConfig(
    dataset_name="beta",
    n_calib_list=[0, 1, 2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
)

df_raw = run_loso_variants_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError("df_raw is empty. Please check data loading / model fitting.")

# ============================================================
# Configuration audit
# ============================================================
# Except for the harmonic-branch ablation itself, every variant should
# report harmonic_n_subbands == 5. The harmonic-branch ablation should
# report harmonic_n_subbands == 0.
audit_cols = [
    "variant",
    "harmonic_n_subbands",
    "enable_stage1",
    "enable_stage2",
    "enable_harmonic_branch",
    "enable_branch_r2",
    "enable_branch_r3",
    "stc_split_mode",
]

df_audit = (
    df_raw[audit_cols]
    .drop_duplicates()
    .sort_values("variant")
)

print("\n========== Configuration Audit ==========")
print(df_audit.to_string(index=False))

unexpected = df_audit[
    (df_audit["enable_harmonic_branch"])
    & (df_audit["harmonic_n_subbands"] != 5)
]
if not unexpected.empty:
    raise RuntimeError(
        "Some harmonic-enabled variants are not using the five-subband "
        "harmonic baseline:\n"
        + unexpected.to_string(index=False)
    )

harmonic_off = df_audit[
    ~df_audit["enable_harmonic_branch"]
]
if not harmonic_off.empty and not (harmonic_off["harmonic_n_subbands"] == 0).all():
    raise RuntimeError(
        "The no-harmonic ablation has an unexpected harmonic_n_subbands value."
    )


# ============================================================
# 9) Compact final summary
# ============================================================
df_summary = (
    df_raw.groupby(
        ["dataset", "variant", "n_calib"],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        n_runs=("acc", "count"),
    )
    .sort_values(["variant", "n_calib"])
)

print("\n========== Ablation Summary ==========")
print(
    df_summary[
        [
            "dataset",
            "variant",
            "n_calib",
            "acc_mean",
            "acc_std",
            "itr_mean",
            "itr_std",
            "n_runs",
        ]
    ].to_string(index=False)
)

Number of harmonic subbands: 5
Harmonic filter weights: [0.38648818 0.20729625 0.15560897 0.13195532 0.11865127]
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_w

In [5]:
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """
    ITR bits/min
    acc: 准确率[0,1]
    n_classes: 类别数
    t_sec: 单次决策时间(秒)
    """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(dataset=dataset, subjects=[s], return_concat=True)
        X = np.asarray(X)
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


# =========================
# 全体被试 LOSO（不分组）
# =========================
# 你需要提前定义：
# all_subjects, sfreq, freq_list, BETA_paradigm, BETA_dataset, SUTLSSVEP

n_classes = 40   # 按你的数据集类别数设置
T = 1.0          # 单次决策时间(秒)，按实验窗口修改

print("\n========== LOSO on ALL subjects ==========")
print(f"n_classes={n_classes}, T={T}s")

all_data = build_subject_data(BETA_paradigm, BETA_dataset, all_subjects)

all_results = []
accs, itrs = [], []

for target_sid in all_subjects:
    source_sids = [s for s in all_subjects if s != target_sid]
    source_data = {s: all_data[s] for s in source_sids}
    X_t, y_t = all_data[target_sid]

    model = SUTLSSVEP(
        srate=sfreq,
        freqs=[float(f) for f in freq_list],
        n_harmonics=3,
        n_bands=5,
        top_m1=20,
        n_jobs=8
    )

    model.fit(source_data)
    y_pred = model.predict(X_t)

    acc = float(np.mean(y_pred == y_t))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=T))

    accs.append(acc)
    itrs.append(itr)

    print(f"target S{target_sid}: acc={acc:.4f}, itr={itr:.2f} bits/min")

    all_results.append({
        "subject_id": target_sid,
        "subject": f"S{target_sid}",
        "acc": acc,
        "itr": itr,
        "n_trials": len(y_t)
    })

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


# =========================
# 最终汇总打印（仅打印，不保存csv）
# =========================
df = pd.DataFrame(all_results).sort_values("subject_id")

print("\n========== Subject-wise Results ==========")
print(df.to_string(index=False))

print("\n========== Overall Summary ==========")
summary_df = pd.DataFrame([{
    "acc_mean": df["acc"].mean(),
    "acc_std": df["acc"].std(),
    "itr_mean(bits/min)": df["itr"].mean(),
    "itr_std(bits/min)": df["itr"].std(),
    "n_subjects": df["subject_id"].nunique()
}])
print(summary_df.to_string(index=False))

print(f"\nOverall mean acc = {df['acc'].mean():.4f}")
print(f"Overall mean itr = {df['itr'].mean():.2f} bits/min")


========== LOSO on ALL subjects ==========
n_classes=40, T=1.0s
target S1: acc=0.6875, itr=166.45 bits/min
target S2: acc=0.7750, itr=201.81 bits/min
target S3: acc=0.7812, itr=204.47 bits/min
target S4: acc=0.4313, itr=79.77 bits/min
target S5: acc=0.5312, itr=110.83 bits/min
target S6: acc=0.3812, itr=65.56 bits/min
target S7: acc=0.4062, itr=72.55 bits/min
target S8: acc=0.4188, itr=76.14 bits/min
target S9: acc=0.5500, itr=117.04 bits/min
target S10: acc=0.4562, itr=87.21 bits/min
target S11: acc=0.1750, itr=17.55 bits/min
target S12: acc=0.6875, itr=166.45 bits/min
target S13: acc=0.5563, itr=119.14 bits/min
target S14: acc=0.5625, itr=121.25 bits/min
target S15: acc=0.5938, itr=132.01 bits/min
target S16: acc=0.4375, itr=81.61 bits/min
target S17: acc=0.2875, itr=41.44 bits/min
target S18: acc=0.9437, itr=282.73 bits/min
target S19: acc=0.5125, itr=104.74 bits/min
target S20: acc=0.4125, itr=74.34 bits/min
target S21: acc=0.5437, itr=114.96 bits/min
target S22: acc=0.5563, itr=1

In [4]:
# IISTLF
def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(
            dataset=dataset,
            subjects=[s],
            return_concat=True,
            verbose=False,
        )
        X = np.nan_to_num(
            np.asarray(X, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


def pick_per_stimulus_trials(
    X,
    y,
    n_classes,
    n_calib_per_stimulus=2,
    random_state=42,
):
    """每个刺激类别抽取相同数量的校准 trial。"""
    rng = np.random.RandomState(random_state)
    selected = []

    for cls in range(n_classes):
        idx_cls = np.flatnonzero(y == cls)
        if len(idx_cls) < n_calib_per_stimulus:
            raise ValueError(
                f"目标被试类别 {cls} trial 不足: "
                f"{len(idx_cls)} < {n_calib_per_stimulus}"
            )
        selected.extend(
            rng.choice(
                idx_cls,
                size=n_calib_per_stimulus,
                replace=False,
            )
        )

    selected = np.sort(np.asarray(selected, dtype=int))
    mask = np.ones(len(y), dtype=bool)
    mask[selected] = False

    X_cal = X[selected]
    y_cal = y[selected]
    X_test = X[mask]
    y_test = y[mask]

    if len(X_test) == 0:
        raise ValueError("校准后没有剩余测试 trial。")
    if np.intersect1d(selected, np.flatnonzero(mask)).size:
        raise AssertionError("calibration/test selections overlap")
    if any(np.sum(y_cal == cls) != n_calib_per_stimulus for cls in range(n_classes)):
        raise AssertionError("每个刺激类别的校准 trial 数量不一致")

    return X_cal, y_cal, X_test, y_test


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 主评估：全体 LOSO（不分组）
# =========================
all_subjects = list(range(1, 71))
n_calib_per_stimulus = 1
random_state = 42

n_classes = len(freq_list)
t_sec = 1.0

print("\n========== LOSO on ALL subjects (IISTLF) ==========")
print(f"n_classes={n_classes}, t_sec={t_sec}")

all_data = build_subject_data(BETA_paradigm, BETA_dataset, all_subjects)
all_results = []
accs = []
itrs = []

for target_sid in all_subjects:
    X_t, y_t = all_data[target_sid]

    X_cal, y_cal, X_test, y_test = pick_per_stimulus_trials(
        X_t, y_t,
        n_classes=n_classes,
        n_calib_per_stimulus=n_calib_per_stimulus,
        random_state=random_state,
    )

    source_sids = [s for s in all_subjects if s != target_sid]
    src_accs = []

    for s_sid in source_sids:
        X_s, y_s = all_data[s_sid]

        model = IISTLF(
            srate=sfreq,
            freqs=[float(f) for f in freq_list],
            n_harmonics=3,
            n_subbands=5,          # 与 Table 1 披露的 IISTLF 子带数一致
        )
        model.fit(
            source_Xy=(X_s, y_s.astype(int)),
            target_calib_X=X_cal,
            target_calib_y=y_cal,
        )
        #   y_pred, scores = model.predict(X_test, return_scores=True)
        y_pred = model.predict(X_test)
        src_accs.append(float(np.mean(y_pred == y_test)))

    acc = float(np.mean(src_accs))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=t_sec))

    accs.append(acc)
    itrs.append(itr)

    print(
        f"target S{target_sid}: acc={acc:.4f}, "
        f"itr={itr:.2f} bits/min "
        f"(avg over {len(source_sids)} sources)"
    )

    all_results.append(
        {
            "subject_id": target_sid,
            "subject": f"S{target_sid}",
            "acc": acc,
            "itr": itr,
        }
    )

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


========== LOSO on ALL subjects (IISTLF) ==========
n_classes=40, t_sec=1.0
target S1: acc=0.6909, itr=167.78 bits/min (avg over 69 sources)
target S2: acc=0.7225, itr=180.18 bits/min (avg over 69 sources)
target S3: acc=0.7157, itr=177.48 bits/min (avg over 69 sources)
target S4: acc=0.4365, itr=81.31 bits/min (avg over 69 sources)
target S5: acc=0.3707, itr=62.66 bits/min (avg over 69 sources)
target S6: acc=0.4161, itr=75.36 bits/min (avg over 69 sources)
target S7: acc=0.3312, itr=52.24 bits/min (avg over 69 sources)
target S8: acc=0.3237, itr=50.34 bits/min (avg over 69 sources)
target S9: acc=0.5777, itr=126.43 bits/min (avg over 69 sources)
target S10: acc=0.4547, itr=86.75 bits/min (avg over 69 sources)
target S11: acc=0.1839, itr=19.21 bits/min (avg over 69 sources)
target S12: acc=0.6789, itr=163.14 bits/min (avg over 69 sources)
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S31-S40.tar.gz
--------ssssss, /upload/li

In [6]:
# **************************************************
# BETA数据集 eTRCA方法 - 全体数据LOSO实验（不分组，含ITR）
# **************************************************

# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ ITR bits/min """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


# ========== eTRCA 固定参数配置 ==========
srate = 250
T = 1         # 数据时长（秒）
n_harmonics = 3

# 滤波器组配置
wp = [(5, 90)]
ws = [(3, 92)]
filterbank = generate_filterbank(wp, ws, srate=srate, order=15, rp=0.5)

# 获取频率列表与类别数
events = BETA_dataset.events.keys()
freq_list = [BETA_dataset.get_freq(event) for event in events]
n_classes = len(freq_list)   # 通常是40类

# 生成参考信号
Yf = generate_cca_references(freq_list, srate=srate, T=T, n_harmonics=n_harmonics)
print(f"📊 CCA reference shape: {Yf.shape}")
print(f"📊 Number of classes: {n_classes}")
print(f"📊 Decision window t_sec: {T}")

# ========== 一次性加载全体数据（不分组） ==========
all_subjects = list(range(1, 71))
X_all, y_all, meta_all = BETA_paradigm.get_data(
    BETA_dataset,
    subjects=all_subjects ,          # 全体被试
    return_concat=True,
    n_jobs=None,
    verbose=False
)

print("\n" + "#" * 70)
print("# 🔬 LOSO on ALL subjects")
print("#" * 70)
print(f"📐 Data shape: {X_all.shape}")

all_subjects = np.unique(meta_all['subject'])
N = len(all_subjects)

print(f"👥 Total subjects: {N}")
print(f"📋 Subject IDs: {all_subjects}")

# ========== 存储结果 ==========
acc_list = []
itr_list = []
subject_results = []

# ========== LOSO循环 ==========
for i, test_subject in enumerate(all_subjects):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    # LOSO划分：当前被试测试，其余训练
    test_mask = (meta_all['subject'] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"📊 Training samples: {len(train_ind)}")
    print(f"📊 Testing samples: {len(test_ind)}")

    # 初始化SC_TRCA模型
    estimator = SC_TRCA(
        standard=False,   # 非标准TRCA
        ensemble=True,    # ensemble
        n_components=1
    )

    # 训练
    print("🔄 SC_TRCA训练中...")
    estimator.fit(X_train, y_train, Yf)
    print("✅ SC_TRCA训练完成")

    # 预测（SC_TRCA返回(相关系数, 标签)）
    _, p_labels = estimator.predict(X_test)

    # 指标计算
    n_correct = int(np.sum(p_labels == y_test))
    acc_s = n_correct / len(y_test)
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=T))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    subject_results.append({
        "test_subject": int(test_subject),
        "accuracy": float(acc_s),
        "itr": float(itr_s),
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# ========== 汇总统计 ==========
acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

df_subject = pd.DataFrame(subject_results).sort_values("test_subject")

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_subject.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)


📊 CCA reference shape: (40, 6, 250)
📊 Number of classes: 40
📊 Decision window t_sec: 1
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------

In [7]:
# **************************************************
# BETA数据集 TtCCA - 全体数据LOSO实验（不分组，含ITR）
# **************************************************

# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ ITR bits/min """
    p = np.clip(acc, eps, 1 - eps)
    term = np.log2(n_classes) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    return term * 60.0 / t_sec


# ========== 固定参数 ==========
srate = 250
T = 1
n_harmonics = 3

events = BETA_dataset.events.keys()
freq_list = [BETA_dataset.get_freq(event) for event in events]
n_classes = len(freq_list)

# 生成CCA参考信号模板
cca_template = generate_cca_references(freq_list, srate=srate, T=T, n_harmonics=n_harmonics)
print(f"📊 CCA template shape: {cca_template.shape}")
print(f"📊 Number of classes: {n_classes}")
print(f"📊 Decision time: {T}s")


# ========== 一次性加载全体数据（不分组） ==========
all_subjects = list(range(1, 71))
X_all, y_all, meta_all = BETA_paradigm.get_data(
    BETA_dataset,
    subjects=all_subjects,          # 全体被试
    return_concat=True,
    n_jobs=None,
    verbose=False
)

print(f"\n📐 Data shape: {X_all.shape}")

all_subjects = np.unique(meta_all['subject'])
N = len(all_subjects)
print(f"👥 Total subjects: {N}")
print(f"📋 Subject IDs: {all_subjects}")

# ========== LOSO ==========
acc_list = []
itr_list = []
subject_results = []

for i, test_subject in enumerate(all_subjects):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    # LOSO划分
    test_mask = (meta_all['subject'] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"📊 Training samples: {len(train_ind)}")
    print(f"📊 Testing samples: {len(test_ind)}")

    # 初始化TtCCA模型
    estimator = TtCCA(
        n_components=1,
        n_jobs=-1
    )

    # 训练
    print("🔄 TtCCA训练中...")
    estimator.fit(X_train, y_train, cca_template, n_harmonics)
    print("✅ TtCCA训练完成")

    # 预测
    p_labels = estimator.predict(X_test)

    # 指标
    n_correct = int(np.sum(p_labels == y_test))
    acc_s = n_correct / len(y_test)
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=T))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    subject_results.append({
        "test_subject": int(test_subject),
        "accuracy": float(acc_s),
        "itr": float(itr_s),
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# ========== 汇总 ==========
acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

df_results = pd.DataFrame(subject_results).sort_values("test_subject")

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)


📊 CCA template shape: (40, 6, 250)
📊 Number of classes: 40
📊 Decision time: 1s
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, 

In [5]:
# noise test
# ============================================================
# 1) Utilities
# ============================================================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR in bits/min."""
    p = np.clip(acc, eps, 1.0 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1.0 - p) * np.log2((1.0 - p) / (n_classes - 1))
    )
    return float(term * 60.0 / t_sec)


def add_awgn_at_snr(X, snr_db, rng, eps=1e-12):
    """
    按每个 trial 的信号功率加入 AWGN。

    X:      (n_trials, n_channels, n_samples)
    snr_db: 目标 SNR，单位 dB
    rng:    np.random.RandomState
    """
    X = np.asarray(X, dtype=np.float64)

    if snr_db is None:
        return X.copy()

    # 去除每个通道的时间均值后估计信号功率
    X_centered = X - X.mean(axis=-1, keepdims=True)
    signal_power = np.mean(
        X_centered ** 2,
        axis=(1, 2),
        keepdims=True,
    )

    noise_power = signal_power / (10.0 ** (float(snr_db) / 10.0))
    noise_std = np.sqrt(np.maximum(noise_power, eps))
    noise = rng.normal(loc=0.0, scale=1.0, size=X.shape)

    return X + noise * noise_std


def subject_split_trials(X_subj, y_subj, n_calib, rng):
    """
    每个类别抽取 n_calib 个校准 trial，其余 trial 作为测试集。
    """
    X_subj = np.asarray(X_subj)
    y_subj = np.asarray(y_subj)

    calib_idx = []
    test_idx = []

    for cls in np.unique(y_subj):
        idx = np.flatnonzero(y_subj == cls).copy()
        rng.shuffle(idx)

        take = min(int(n_calib), len(idx))
        calib_idx.extend(idx[:take].tolist())
        test_idx.extend(idx[take:].tolist())

    calib_idx = np.asarray(calib_idx, dtype=int)
    test_idx = np.asarray(test_idx, dtype=int)

    return (
        X_subj[calib_idx],
        y_subj[calib_idx],
        X_subj[test_idx],
        y_subj[test_idx],
    )


# ============================================================
# 2) Experiment config
# ============================================================
@dataclass
class VariantConfig:
    name: str
    model_kwargs: Dict[str, Any]


@dataclass
class RunConfig:
    dataset_name: str
    n_calib_list: List[int]
    repeat_seeds: List[int]
    n_classes: int = 40
    decision_time: float = 1.0
    window_sec: float = 1.0

    # 噪声鲁棒性开关
    enable_noise_robustness: bool = True

    # clean 将自动加入；以下为额外 SNR 条件
    noise_snr_list: tuple = (20.0, 10.0, 5.0, 0.0)

    # 推荐 test_only：只污染测试数据
    # 可选 test_and_calibration：同时污染目标校准数据
    noise_scope: str = "test_only"


# ============================================================
# 3) Dataset loader
# ============================================================
def load_dataset_for_loso_by_subjects(dataset_name: str, selected_subjects):
    """加载 beta 数据。相关数据对象需已在 Notebook 中定义。"""
    dataset_name = dataset_name.lower()

    if dataset_name != "beta":
        raise ValueError(f"Unsupported dataset_name: {dataset_name}")

    subjects_data = []
    subjects_label = []
    subject_ids = []

    for sid in selected_subjects:
        X_s, y_s, _ = BETA_paradigm.get_data(
            BETA_dataset,
            subjects=[sid],
            return_concat=True,
            n_jobs=None,
            verbose=False,
        )

        X_s = np.nan_to_num(
            np.asarray(X_s, dtype=np.float64),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y_s = np.asarray(y_s).astype(int)

        subjects_data.append(X_s)
        subjects_label.append(y_s)
        subject_ids.append(sid)

    freqs_float = [float(f) for f in freq_list]
    return subjects_data, subjects_label, subject_ids, freqs_float


# ============================================================
# 4) One target subject under one SNR condition
# ============================================================
def _run_one_target_task(
    tgt_idx,
    n_subj,
    subject_ids,
    subjects_data,
    subjects_label,
    n_calib,
    seed,
    variant,
    base_model_kwargs,
    freqs_float,
    run_cfg,
    snr_db,
):
    rng = np.random.RandomState(seed)

    target_subject_id = subject_ids[tgt_idx]
    X_target = subjects_data[tgt_idx]
    y_target = subjects_label[tgt_idx]

    X_cal, y_cal, X_test, y_test = subject_split_trials(
        X_target,
        y_target,
        n_calib=n_calib,
        rng=rng,
    )

    if len(y_test) == 0:
        return []

    source_X = []
    source_y = []
    source_subjects = []

    for src_idx in range(n_subj):
        if src_idx == tgt_idx:
            continue

        sid = subject_ids[src_idx]
        X_source_subject = subjects_data[src_idx]
        y_source_subject = subjects_label[src_idx]

        source_X.append(X_source_subject)
        source_y.append(y_source_subject)
        source_subjects.append(
            np.full(len(y_source_subject), sid, dtype=int)
        )

    X_source = np.concatenate(source_X, axis=0)
    y_source = np.concatenate(source_y, axis=0)
    subjects_source = np.concatenate(source_subjects, axis=0)

    # 每个 target fold 和 SNR 使用独立但可复现的噪声随机数
    if snr_db == "clean":
        noise_rng = None
        X_cal_model = X_cal
        X_test_eval = X_test
    else:
        noise_seed = (
            int(seed)
            + 1000003 * (int(tgt_idx) + 1)
            + int(round((float(snr_db) + 100.0) * 10.0))
        )
        noise_rng = np.random.RandomState(noise_seed)

        # 默认只污染测试数据，用于独立评估测试阶段噪声鲁棒性
        X_test_eval = add_awgn_at_snr(
            X_test,
            snr_db=float(snr_db),
            rng=noise_rng,
        )

        X_cal_model = X_cal
        if (
            run_cfg.noise_scope == "test_and_calibration"
            and n_calib > 0
        ):
            X_cal_model = add_awgn_at_snr(
                X_cal,
                snr_db=float(snr_db),
                rng=noise_rng,
            )

    if run_cfg.noise_scope not in {
        "test_only",
        "test_and_calibration",
    }:
        raise ValueError(
            "noise_scope must be 'test_only' or 'test_and_calibration'"
        )

    model_kwargs = deepcopy(base_model_kwargs)
    model_kwargs.update(variant.model_kwargs)
    model_kwargs["random_state"] = int(seed)

    if model_kwargs.get("freqs") is None:
        model_kwargs["freqs"] = freqs_float

    model = SQHAF(**model_kwargs)

    model.fit(
        X_source=X_source,
        y_source=y_source,
        subjects_source=subjects_source,
        target_calib_X=X_cal_model if n_calib > 0 else None,
        target_calib_y=y_cal if n_calib > 0 else None,
    )

    calibration_X = X_cal_model if n_calib > 0 else None
    y_pred = model.predict(
        X_test_eval,
        calib_X=calibration_X,
    )

    acc = float(np.mean(y_pred == y_test))
    itr = compute_itr(
        acc,
        n_classes=run_cfg.n_classes,
        t_sec=run_cfg.decision_time,
    )

    return [{
        "dataset": run_cfg.dataset_name,
        "variant": variant.name,
        "target_subject_id": target_subject_id,
        "n_calib": int(n_calib),
        "seed": int(seed),
        "snr_db": snr_db,
        "acc": acc,
        "itr": itr,
        "n_test": int(len(y_test)),
        "noise_scope": run_cfg.noise_scope,
    }]


# ============================================================
# 5) Parallel full LOSO noise evaluation
# ============================================================
def run_loso_noise_robustness_parallel(
    run_cfg,
    variants,
    base_model_kwargs,
    subjects,
    n_jobs=-1,
    backend="loky",
    prefer=None,
    verbose=0,
):
    (
        subjects_data,
        subjects_label,
        subject_ids,
        freqs_float,
    ) = load_dataset_for_loso_by_subjects(
        run_cfg.dataset_name,
        subjects,
    )

    n_subj = len(subjects_data)
    rows = []

    if run_cfg.enable_noise_robustness:
        snr_conditions = ["clean"] + [
            float(snr) for snr in run_cfg.noise_snr_list
        ]
    else:
        snr_conditions = ["clean"]

    for variant in variants:
        for n_calib in run_cfg.n_calib_list:
            seeds = [0] if n_calib == 0 else run_cfg.repeat_seeds

            for seed in seeds:
                for snr_db in snr_conditions:
                    out = Parallel(
                        n_jobs=n_jobs,
                        backend=backend,
                        prefer=prefer,
                        verbose=verbose,
                    )(
                        delayed(_run_one_target_task)(
                            tgt_idx=tgt_idx,
                            n_subj=n_subj,
                            subject_ids=subject_ids,
                            subjects_data=subjects_data,
                            subjects_label=subjects_label,
                            n_calib=n_calib,
                            seed=seed,
                            variant=variant,
                            base_model_kwargs=base_model_kwargs,
                            freqs_float=freqs_float,
                            run_cfg=run_cfg,
                            snr_db=snr_db,
                        )
                        for tgt_idx in range(n_subj)
                    )

                    for result in out:
                        if result:
                            rows.extend(result)

    return pd.DataFrame(rows)


# ============================================================
# 6) Main method only
# ============================================================
variants = [
    VariantConfig(
        "full",
        {
            "source_score_mode": "similarity_confidence",
            "stc_split_mode": "time_ordered",
            "enable_harmonic_branch": True,
        },
    ),
]


# ============================================================
# 7) Run noise robustness experiment
# ============================================================
freqs_float = [float(f) for f in freq_list]
window_sec = 1.0

base_model_kwargs = dict(
    n_sources=10,
    neighbor_radius=1,
    neighbor_decay=0.5,
    source_score_mode="similarity_confidence",
    confidence_lambda=0.2,
    target_alignment_mode="calibration",
    stc_split_mode="time_ordered",
    enable_harmonic_branch=True,
    enable_stage2=True,
    Yf=generate_cca_references(
        freqs=freqs_float,
        srate=sfreq,
        T=window_sec,
        n_harmonics=3,
    ),
)

run_cfg = RunConfig(
    dataset_name="beta",
    n_calib_list=[0,1,2],
    repeat_seeds=list(range(5)),
    n_classes=40,
    decision_time=window_sec,
    window_sec=window_sec,
    enable_noise_robustness=True,
    noise_snr_list=(20.0, 10.0, 5.0, 0.0),
    noise_scope="test_only",
)

df_raw = run_loso_noise_robustness_parallel(
    run_cfg=run_cfg,
    variants=variants,
    base_model_kwargs=base_model_kwargs,
    subjects=all_subjects,
    n_jobs=8,
    backend="loky",
    verbose=0,
)

if df_raw.empty:
    raise RuntimeError(
        "df_raw is empty. Please check data loading / model fitting."
    )


# ============================================================
# 8) Compact robustness summary
# ============================================================
df_summary = (
    df_raw.groupby(
        ["dataset", "variant", "n_calib", "snr_db"],
        as_index=False,
    )
    .agg(
        acc_mean=("acc", "mean"),
        acc_std=("acc", "std"),
        itr_mean=("itr", "mean"),
        itr_std=("itr", "std"),
        n_runs=("acc", "count"),
    )
)

df_summary["_snr_order"] = df_summary["snr_db"].map(
    lambda x: -1.0 if x == "clean" else float(x)
)

df_summary = (
    df_summary
    .sort_values(
        ["dataset", "variant", "n_calib", "_snr_order"],
    )
    .drop(columns="_snr_order")
    .reset_index(drop=True)
)

print("\n========== Noise Robustness Summary ==========")
print(
    df_summary[
        [
            "dataset",
            "variant",
            "n_calib",
            "snr_db",
            "acc_mean",
            "acc_std",
            "itr_mean",
            "itr_std",
            "n_runs",
        ]
    ].to_string(index=False)
)

--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchu

In [6]:
# TDCA
# =========================
# ITR函数
# =========================
def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec

# =========================
# TDCA参数区
# =========================
n_harmonics = 3
window_length = 1      # 时间窗（秒）
ref_T = 0.9          # 比0.5短，避免TDCA长度冲突
padding_len = 1
n_components = 4


n_classes = len(freq_list)
t_sec = window_length

print(f"[BETA] n_classes={n_classes}, t_sec={t_sec:.2f}s")

# 参考信号（注意这里要float频率）
Yf = generate_cca_references(
    [float(f) for f in freq_list],
    srate=sfreq,
    T=ref_T,
    n_harmonics=n_harmonics
)
print(f"[BETA] CCA references shape: {Yf.shape}")

# =========================
# 一次性加载全体数据（LOSO）
# =========================
X_all, y_all, meta_all = BETA_paradigm.get_data(
    BETA_dataset,
    subjects=all_subjects,
    return_concat=True,
    n_jobs=None,
    verbose=False
)

all_subject_ids = np.unique(meta_all["subject"])
N = len(all_subject_ids)

print("\n" + "#" * 70)
print("# 🔬 TDCA LOSO on BETA ALL subjects")
print("#" * 70)
print(f"Total subjects: {N}")
print(f"Subject IDs: {all_subject_ids}")
print(f"Data shape: {X_all.shape}")

# =========================
# LOSO循环
# =========================
acc_list = []
itr_list = []
results = []

for i, test_subject in enumerate(all_subject_ids):
    print(f"\n{'='*50}")
    print(f"🧪 Fold {i+1}/{N}: Testing on Subject {test_subject}")
    print(f"{'='*50}")

    test_mask = (meta_all["subject"] == test_subject)
    train_mask = ~test_mask

    train_ind = np.where(train_mask)[0]
    test_ind = np.where(test_mask)[0]

    X_train, y_train = X_all[train_ind], y_all[train_ind]
    X_test, y_test = X_all[test_ind], y_all[test_ind]

    print(f"Training samples: {len(train_ind)}")
    print(f"Testing samples: {len(test_ind)}")
    print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

    estimator = TDCA(
        padding_len=padding_len,
        n_components=n_components
    )

    # 安全检查，防止长度报错
    x_len = X_train.shape[-1]
    yf_len = Yf.shape[-1]
    if x_len <= (padding_len + yf_len):
        raise ValueError(
            f"Not enough time points for TDCA: "
            f"X_len={x_len}, padding_len={padding_len}, Yf_len={yf_len}, "
            f"need X_len > padding_len + Yf_len.\n"
            f"Try smaller ref_T, smaller padding_len, or longer data window."
        )

    p_labels = estimator.fit(
        X=X_train,
        y=y_train,
        Yf=Yf
    ).predict(X_test)

    acc_s = float(np.mean(p_labels == y_test))
    itr_s = float(compute_itr(acc_s, n_classes=n_classes, t_sec=t_sec))

    acc_list.append(acc_s)
    itr_list.append(itr_s)

    n_correct = int(np.sum(p_labels == y_test))
    print(f"✅ Subject {test_subject} Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%) | ITR: {itr_s:.2f} bits/min")

    results.append({
        "test_subject": int(test_subject),
        "accuracy": acc_s,
        "itr": itr_s,
        "n_test_samples": int(len(test_ind)),
        "n_correct": n_correct
    })

# =========================
# 汇总打印
# =========================
df_results = pd.DataFrame(results).sort_values("test_subject")

acc_array = np.array(acc_list)
itr_array = np.array(itr_list)

mean_acc = float(acc_array.mean())
std_acc = float(acc_array.std())
mean_itr = float(itr_array.mean())
std_itr = float(itr_array.std())

print("\n" + "=" * 70)
print("📋 Subject-wise Results")
print("=" * 70)
print(df_results.to_string(index=False))

print("\n" + "=" * 70)
print("📊 Overall LOSO Summary")
print("=" * 70)
print(f"Mean ± Std Accuracy: {mean_acc:.4f} ± {std_acc:.4f} ({mean_acc*100:.2f}% ± {std_acc*100:.2f}%)")
print(f"Mean ± Std ITR:      {mean_itr:.2f} ± {std_itr:.2f} bits/min")
print(f"Number of subjects: {N}")

print("\n" + "#" * 70)
print("# 🎉 All experiments completed!")
print("#" * 70)

[BETA] n_classes=40, t_sec=1.00s
[BETA] CCA references shape: (40, 6, 225)
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upl

In [10]:
# EEGconformer
def build_subject_data(paradigm, dataset, subjects):
    """
    返回 dict[subject_id] = (X, y)
    """
    sub_data = {}
    for s in subjects:
        X, y, meta = paradigm.get_data(
            dataset=dataset,
            subjects=[s],
            return_concat=True,
            verbose=False,
        )
        X = np.nan_to_num(
            np.asarray(X, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
        y = np.asarray(y).astype(int)
        sub_data[s] = (X, y)
    return sub_data


def compute_itr(acc, n_classes=40, t_sec=1.0, eps=1e-12):
    """ITR bits/min"""
    p = np.clip(acc, eps, 1 - eps)
    term = (
        np.log2(n_classes)
        + p * np.log2(p)
        + (1 - p) * np.log2((1 - p) / (n_classes - 1))
    )
    return term * 60.0 / t_sec


# =========================
# 主评估：全体 LOSO（无目标校准、无目标微调）
# =========================
all_subjects = list(range(1, 71))
random_state = 42
n_classes = len(freq_list)
t_sec = 1.0

print("\n========== LOSO on ALL subjects (EEG-Conformer) ==========")
print(f"n_classes={n_classes}, t_sec={t_sec}")

all_data = build_subject_data(BETA_paradigm, BETA_dataset, all_subjects)
accs = []
itrs = []

for target_sid in all_subjects:
    X_test, y_test = all_data[target_sid]
    train_sids = [s for s in all_subjects if s != target_sid]
    X_train = np.concatenate([all_data[s][0] for s in train_sids], axis=0)
    y_train = np.concatenate([all_data[s][1] for s in train_sids], axis=0)

    model = EEGConformer(
        srate=sfreq,
        freqs=[float(f) for f in freq_list],
        device="cuda" if torch.cuda.is_available() else "cpu",
        emb_size=40,
        depth=3,
        n_heads=4,
        batch_size=256,
        n_epochs=100,
        lr=3e-4,
        beta1=0.5,
        beta2=0.999,
        seed=random_state,
        use_amp=True,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test, batch_size=512)

    acc = float(np.mean(y_pred == y_test))
    itr = float(compute_itr(acc, n_classes=n_classes, t_sec=t_sec))
    accs.append(acc)
    itrs.append(itr)

    print(f"target S{target_sid}: acc={acc:.4f}, itr={itr:.2f} bits/min")

print(f"\nLOSO mean acc = {np.mean(accs):.4f}")
print(f"LOSO mean itr = {np.mean(itrs):.2f} bits/min")


========== LOSO on ALL subjects (EEG-Conformer) ==========
n_classes=40, t_sec=1.0
target S1: acc=0.4375, itr=81.61 bits/min
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S31-S40.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S1-S10.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S11-S20.tar.gz
--------ssssss, /upload/liubingchuan_BETA_wof/S41-S50.tar.gz
--------ssssss, /uploa